# Manually Test chFPGA Firmware for CRS Platform, Pocket Correlator Mode

A notebook used to test the chFPGA firmware for the CRS platform in Pocket Correlator mode. The tests are conducted "manually" in this notebook, meaning that the data products are captured and processed using `corriscope` methods. The signal processing developed in this script is then used to write wrapper functions in `pocket_correlator.py`, which provides a higher-level interface with the pocket correlator. The functions of `pocket_correlator.py` are then tested in the notebook `test_pocket_correlator_auto.ipynb`. 

Revisions:
- **October 13, 2025**: includes changes to accomodate a big UCORR update, allowing for software integration methods that have been unavailable since June 2025. Also includes updates for double-resolution FFT capture.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 12})
import os

# Program and Connect to the CRS Board

In [ ]:
run ../../../corriscope/pocket_correlator.py --serial 0050 --stderr_log_level debug --prog 2

In [ ]:
pc.i.GPIO.NUMBER_OF_CHANNELIZERS_WITH_FFT

In [ ]:
pc.i.GPIO.NUMBER_OF_CHANNELIZERS

## Set Up a Signal Generator (If Desired)

The following cell requires labpy (https://bitbucket.org/winterlandcosmology/labpy/src/jfc_dev/) to be cloned and installed in your virtual environment. The jfc_dev branch is recommended.

If the signal generator is not used, it is assumed that another signal is being injected into the ADC, such as amplified noise from a 50 $\Omega$ load.

In [ ]:
use_sg = False # Do not use signal generator
# use_sg = True # Use signal generator

if use_sg:
    pwd = os.getcwd()
    os.chdir('/home/ih/labpy')
    from labpy.instruments.rigol.rigol_dsg800 import RigolDSG800
    os.chdir(pwd)
    
    sg = RigolDSG800(adapter = 'TCPIP::10.10.10.152') # DSG836
    # f_carrier = pc.f_res * 1311 # MHz | 573.12986341e6 # 241 * 1e6
    f_carrier = 1000*pc.f_res # 1*pc.f_res + pc.fs # MHz | 573.12986341e6 # 241 * 1e6
    # f_carrier = 1*pc.f_res + pc.fs # MHz
    print(f'f_carrier = {f_carrier} MHz')
    # sg.set_frequency(f'{f_carrier*1e6}')
    sg.set_frequency(f'{f_carrier*1e6}')
    sg.set_output_enable(enable = True)
    power_carrier = -30
    sg.set_amplitude(ampl = power_carrier, units = 'dBm')

# Capture ADC Data

In [ ]:
n_cap_adc = 1000

In [ ]:
# # Check to make sure we're configured to receive channelizer data, not correlated data:
# if pc.ucap.OUTPUT_SOURCE_SEL != 0:
#     pc.ucap.OUTPUT_SOURCE_SEL = 0

# # Make sure we can get data from all channels:
# if pc.ucap.MODE != 0:
#     pc.ucap.MODE = 0 

# # Check that the FFT is not bypassed, the SCALER is bypassed, and set the data source as desired
# for n in range(pc.NPOLS):
#     pc.i.chan[n].FUNCGEN.set_data_source('adc')
#     pc.i.chan[n].FFT.BYPASS = 1
#     pc.i.chan[n].SCALER.BYPASS = 1

# # Start data capture, grabbing data at the output of the
# # scaler, which has been bypassed.
# pc.i.start_data_capture(period = 0.02, 
#                          source = 'adc'
#                          )

# # Capture data!
# t, d_adc, c = pc.u_receiver.read_raw_frames(format = '16', 
#                                              ncap = n_cap_adc//pc.n_frames_per_capture(), 
#                                              split = True, 
#                                              verbose = 0
#                                              )

# d_adc = d_adc >> 2 # shift to 14 bits

In [ ]:
pc.i.set_channelizer(
                    adc_mode=None,
                    adc_sampling_mode=None,
                    adc_bandwidth=None,
                    adcdaq_mode=None,
                    data_source='adc',
                    function=None,
                    funcgen_left_shift=0,
                    funcgen_right_shift=0,
                    # freq_test_bins=None,  # will be passed into function_kwargs
                    fft_bypass=None,
                    fft_shift=None,
                    scaler_bypass=None,
                    scaler_cap_data_type=None,
                    scaler_bypass_data_type=None,
                    postscaler=None,
                    scaler_eight_bit=None,
                    scaler_rounding_mode=None,
                    symmetric_saturation=None,
                    zero_on_sat=None,
                    prober_user_flags=None,
                    offset_binary_encoding=None,
                    local_sync=True,
                    channels=list(np.arange(pc.NPOLS)),
                    gains=None,
                    )

In [ ]:
# Start data capture, grabbing data at the output of the
# scaler, which has been bypassed.
pc.i.start_data_capture(period = 0.02, 
                        source = 'funcgen',
                        mode = pc.ucap.MODE,
                        channels = list(np.arange(pc.NPOLS)),
                        select = True,
                        data_type = None
                        )

# Capture data!
t, d_adc, c = pc.u_receiver.read_raw_frames(format = '16', 
                                            ncap = n_cap_adc//pc.n_frames_per_capture(), 
                                            split = True, 
                                            verbose = 0
                                            )

d_adc = d_adc >> 2 # shift to 14 bits

- The shape of the ADC data in mode 0 is $(N_{\text{pol}}, 2n_{\text{cap}}, M)$, where $N_{\text{pol}} = 8$, $n_{\text{cap}}$ is the number of captures, and $M = 2^{14}$ is the number of samples per frame.
- We receive $2n_{\text{cap}}$ total frames per pol per capture in mode 0, as we get two frames per burst.

In [ ]:
d_adc.shape

In [ ]:
np.std(d_adc[0].flatten())

In [ ]:
adc_rms = np.array([np.std(d_adc[n].flatten()) for n in range(pc.NPOLS)])
power_adc = 10*np.log10((adc_rms*(1/2**14))**2/100/1e-3)
power_adc

In [ ]:
pc.i.GPIO.ADC_OV

In [ ]:
pc.i.GPIO.ADC_OR

In [ ]:
d_adc_psd = np.mean(np.abs(np.fft.rfft(d_adc, axis = 2)), axis = 1)**2

In [ ]:
plt.figure(figsize  = (12, 18))

for n in range(pc.NPOLS):

    plt.subplot(pc.NPOLS, 2, 2*n + 1)
    plt.title(f'ADC{n + 1}, {np.round(power_adc[n], 1)} dBm at ADC')
    plt.plot(np.arange(2**14), d_adc[n, 0], color = 'blue') #, marker = '.', s = 25)
    plt.xlim(0, 256)
    plt.ylabel('$V$ [LSBs]')
    plt.xlabel('sample')
    plt.grid()
    
    plt.subplot(pc.NPOLS, 2, 2*n + 2)
    plt.title(f'ADC{n + 1}, NumPy FFT')
    plt.semilogy(pc.f, d_adc_psd[n][:-1], color = 'black')
    plt.ylabel('power [arb.]')
    plt.xlabel('frequency [MHz]')
    plt.xlim(0, 1600)
    plt.grid()

plt.tight_layout()

# Capture FFT Data

In [ ]:
# Set FFT shift schedule --- note that the shift schedule does nothing in the bitgrowth FFT

for n in range(pc.NPOLS):
    pc.i.chan[n].FFT.FFT_SHIFT = 0b00000000000000 # with bitgrowth FFT, we don't need any shifting, so set the shift schedule to 0
    print(n, pc.i.chan[n].FFT.OVERFLOW_COUNT)

In [ ]:
n_cap_fft = n_cap_adc
n_cap_fft

In [ ]:
# Old, does not compensate for JF's additional scrambling

# def get_casper_fft_descramble(n_bit_fft, n_bit_parallel):
#     """
#     Get the descramble map for a CASPER FFT with
#     2**n_bit_fft channels, presenting 2**n_bit_parallel
#     on each cycle
#     """
#     n_fft = 2**n_bit_fft
#     n_parallel = 2**n_bit_parallel
#     return np.arange(n_fft).reshape(n_fft // n_parallel, n_parallel).transpose().flatten()

# bin_map = get_casper_fft_descramble(13,2)

# Unscrambling that accomodates both JF's and CASPER's scrambling

def make_bin_map():
    f_nat = np.arange(8192)
    f_rev = (f_nat>>11) | ((f_nat & 0b11111111111) << 2)
    frev2 = (f_rev & 0b1111111111100) | ((f_rev + np.arange(8192)) & 0b11)
    return frev2

def make_inv_bin_map():
    return np.argsort(make_bin_map())

bin_map = make_bin_map()
inv_bin_map = make_inv_bin_map()

## Capture Firmware FFT Data, Upper 16+16 Bits

In [ ]:
# # Check to make sure we're configured to receive channelizer data, not correlated data:
# if pc.ucap.OUTPUT_SOURCE_SEL != 0:
#     pc.ucap.OUTPUT_SOURCE_SEL = 0

# # Make sure we can get data from all channels:
# if pc.ucap.MODE != 0:
#     pc.ucap.MODE = 0 

# # Check that the FFT is not bypassed, the SCALER is bypassed, and set the data source as desired
# for n in range(pc.NPOLS):
#     pc.i.chan[n].FUNCGEN.set_data_source('adc')
#     pc.i.chan[n].FFT.BYPASS = 0
    
#     pc.i.chan[n].SCALER.BYPASS = 1
#     # pc.i.chan[n].SCALER.DATA_TYPE = 0 # (old) send the most significant bits of the raw FFT values
#     pc.i.chan[n].SCALER.CAP_DATA_TYPE = 1 # 16+16 MSBs of the raw FFT values

# pc.i.set_gains(2**18)

# pc.i.start_data_capture(period = 0.02, 
#                         source = 'scaler',
#                         mode = pc.ucap.MODE,
#                         channels = list(np.arange(pc.NPOLS)),
#                         data_type = 'gain_fft',
#                         select = True
#                         )

# # Capture data!
# t, d_fft_16_16, c = pc.u_receiver.read_raw_frames(format = '16+16', 
#                                                      ncap = n_cap_fft//pc.n_frames_per_capture(), 
#                                                      split = True,
#                                                      verbose = 0
#                                                      )

In [ ]:
pc.i.set_channelizer(
                    adc_mode=None,
                    adc_sampling_mode=None,
                    adc_bandwidth=None,
                    adcdaq_mode=None,
                    data_source='adc',
                    function=None,
                    funcgen_left_shift=0,
                    funcgen_right_shift=0,
                    # freq_test_bins=None,  # will be passed into function_kwargs
                    fft_bypass=False,
                    fft_shift=0b00000000000000,
                    scaler_bypass=True,
                    scaler_cap_data_type=1,
                    scaler_bypass_data_type=None,
                    postscaler=None,
                    scaler_eight_bit=None,
                    scaler_rounding_mode=None,
                    symmetric_saturation=None,
                    zero_on_sat=None,
                    prober_user_flags=None,
                    offset_binary_encoding=None,
                    local_sync=True,
                    channels=list(np.arange(pc.NPOLS)),
                    gains = None # 1*2**32
                    )

In [ ]:
pc.i.get_fft_bypass()

In [ ]:
pc.i.get_scaler_bypass()

In [ ]:
pc.i.start_data_capture(period = 0.02, 
                        source = 'scaler',
                        mode = pc.ucap.MODE,
                        channels = list(np.arange(pc.NPOLS)),
                        data_type = 1, # 'gain_fft',
                        select = True
                        )

In [ ]:
pc.i.get_scaler_bypass()

In [ ]:
[pc.i.chan[n].SCALER.CAP_DATA_TYPE for n in range(pc.NPOLS)]

In [ ]:
# Capture data!
t, d_fft_16_16, c = pc.u_receiver.read_raw_frames(format = '16+16', 
                                                  ncap = n_cap_fft//pc.n_frames_per_capture(), 
                                                  split = True,
                                                  verbose = 0
                                                  )

In [ ]:
d_fft_16_16.shape

In [ ]:
plt.figure(figsize = (12, 10))
plt.suptitle('raw FFT, 16+16 bits')

for n in range(pc.NPOLS):

    plt.subplot(8, 3, 3*n + 1)
    if n == 0:
        plt.title('real')
    plt.plot(pc.f, d_fft_16_16[n, 0, bin_map].real)
    plt.xticks(np.arange(0, 1600 + 400, 400))
    plt.xlim(0, 1600)
    plt.ylabel(f'ADC{n + 1}')
    plt.grid()

    if n + 1 == 8:
        plt.xlabel('frequency [MHz]')
    
    plt.subplot(8, 3, 3*n + 2)
    if n == 0:
        plt.title('imaginary')
    plt.plot(pc.f, d_fft_16_16[n, 0, bin_map].imag, color = 'C1')
    plt.xticks(np.arange(0, 1600 + 400, 400))
    plt.xlim(0, 1600)
    plt.grid()

    if n + 1 == 8:
        plt.xlabel('frequency [MHz]')
    
    plt.subplot(8, 3, 3*n + 3)
    if n == 0:
        plt.title('power')
    plt.semilogy(pc.f, np.mean(np.abs(d_fft_16_16[n, :, bin_map])**2, axis = 1), color = 'C2')
    plt.xticks(np.arange(0, 1600 + 400, 400))
    plt.xlim(0, 1600)
    plt.grid()

    if n + 1 == 8:
        plt.xlabel('frequency [MHz]')

plt.tight_layout()

In [ ]:
plt.figure(figsize = (12, 4))
plt.title('raw FFT, 16+16 bits, ADC1')
# plt.plot(pc.f, d_fft_16_16[0, 0, bin_map].real)
# plt.plot(pc.f, d_fft_16_16[0, 0, bin_map].imag)
plt.semilogy(pc.f, np.mean(np.abs(d_fft_16_16[0, :, bin_map])**2, axis = 1), color = 'C2')
# plt.semilogy(pc.f, np.mean(np.abs(d_fft_16_16[0])**2, axis = 0), color = 'C2')
# plt.ylim(-8, 8)
# plt.xlim(275, 300)
plt.ylabel('power [arb.]')
plt.xlabel('frequency [MHz]')
plt.grid()
plt.tight_layout()

## Capture Firmware FFT Data, Full 32+32 Bits

In [ ]:
# =========
# OLD
# =========

# # Check to make sure we're configured to receive channelizer data, not correlated data:
# if pc.ucap.OUTPUT_SOURCE_SEL != 0:
#     pc.ucap.OUTPUT_SOURCE_SEL = 0

# # Make sure we can get data from all channels:
# if pc.ucap.MODE != 0:
#     pc.ucap.MODE = 0 

# # Check that the FFT is not bypassed, the SCALER is bypassed, and set the data source as desired
# for n in range(pc.NPOLS):
#     pc.i.chan[n].FUNCGEN.set_data_source('adc')
#     # if pc.i.chan[n].FFT.BYPASS:
#     pc.i.chan[n].FFT.BYPASS = 0
    
#     # if not pc.i.chan[n].SCALER.BYPASS:
#     pc.i.chan[n].SCALER.BYPASS = 1
#     # pc.i.chan[n].SCALER.DATA_TYPE = 2 # even bins
#     pc.i.chan[n].SCALER.CAP_DATA_TYPE = 4 # even bins of raw 32+32-bit FFT 

# # Start data capture, grabbing data at the output of the
# # scaler, which has been bypassed.
# pc.i.start_data_capture(period = 0.02, 
#                          source = 'scaler'
#                          )

# # Capture data!
# t, d_fft_even, c = pc.u_receiver.read_raw_frames(format = '32+32', 
#                                                   ncap = n_cap_fft, 
#                                                   split = True,
#                                                   verbose = 0
#                                                   )

# # Check to make sure we're configured to receive channelizer data, not correlated data:
# if pc.ucap.OUTPUT_SOURCE_SEL != 0:
#     pc.ucap.OUTPUT_SOURCE_SEL = 0

# # Make sure we can get data from all channels:
# if pc.ucap.MODE != 0:
#     pc.ucap.MODE = 0 

# # Check that the FFT is not bypassed, the SCALER is bypassed, and set the data source as desired
# for n in range(pc.NPOLS):
#     pc.i.chan[n].FUNCGEN.set_data_source('adc')
#     # if pc.i.chan[n].FFT.BYPASS:
#     pc.i.chan[n].FFT.BYPASS = 0
    
#     # if not pc.i.chan[n].SCALER.BYPASS:
#     pc.i.chan[n].SCALER.BYPASS = 1
#     # pc.i.chan[n].SCALER.DATA_TYPE = 0 # Send the most significant bits of the raw FFT values
#     # pc.i.chan[n].SCALER.DATA_TYPE = 2 # even bins
#     # pc.i.chan[n].SCALER.DATA_TYPE = 3 # odd bins
#     pc.i.chan[n].SCALER.CAP_DATA_TYPE = 4 # even bins of raw 32+32-bit FFT

# # Start data capture, grabbing data at the output of the
# # scaler, which has been bypassed.
# pc.i.start_data_capture(period = 0.02, 
#                          source = 'scaler'
#                          )

# # Capture data!
# t, d_fft_odd, c = pc.u_receiver.read_raw_frames(format = '32+32', 
#                                                 ncap = n_cap_fft, 
#                                                 split = True,
#                                                 verbose = 0
#                                                 )

In [ ]:
# =========
# OLD
# =========

# # Check to make sure we're configured to receive channelizer data, not correlated data:
# if pc.ucap.OUTPUT_SOURCE_SEL != 0:
#     pc.ucap.OUTPUT_SOURCE_SEL = 0

# # Make sure we can get data from all channels:
# if pc.ucap.MODE != 0:
#     pc.ucap.MODE = 0 

# # Bypass the FFT and the SCALER first. Set the data source as desired
# for n in range(pc.NPOLS):
#     pc.i.chan[n].FUNCGEN.set_data_source('adc')
#     pc.i.chan[n].FFT.BYPASS = 0
    
#     pc.i.chan[n].SCALER.BYPASS = 1
#     # pc.i.chan[n].SCALER.DATA_TYPE = 2 
#     pc.i.chan[n].SCALER.CAP_DATA_TYPE = 4 # even bins of raw 32+32-bit FFT

# # Start data capture, grabbing data at the output of the
# # scaler, which has been bypassed.
# pc.i.start_data_capture(period = 0.02, 
#                         source = 'scaler'
#                         )

# # Capture data!
# t, d_fft_even, c = pc.u_receiver.read_raw_frames(format = '32+32', 
#                                                  ncap = n_cap_fft//pc.n_frames_per_capture(), 
#                                                  split = True,
#                                                  verbose = 0
#                                                  )

# # ====
# # Now do the same for the odd bins, switching Scaler modes:

# # Bypass the FFT and the SCALER first. Set the data source as desired
# for n in range(pc.NPOLS):
#     # pc.i.chan[n].SCALER.DATA_TYPE = 3 
#     pc.i.chan[n].SCALER.CAP_DATA_TYPE = 5 # odd bins of raw 32+32-bit FFT

# # Start data capture, grabbing data at the output of the
# # scaler, which has been bypassed.
# pc.i.start_data_capture(period = 0.02, 
#                          source = 'scaler'
#                          )

# # Capture data!
# t, d_fft_odd, c = pc.u_receiver.read_raw_frames(format = '32+32', 
#                                                       ncap = n_cap_fft//pc.n_frames_per_capture(), 
#                                                       split = True,
#                                                       verbose = 0
#                                                       )

# # Stitch together even and odd bins
# d_fft = np.zeros((d_fft_even.shape[0], d_fft_even.shape[1], 8192), dtype = np.complex128)
# d_fft[..., ::2] = d_fft_even
# d_fft[..., 1::2] = d_fft_odd
# d_fft_mean_psd = np.mean(np.abs(d_fft)**2, axis = 1)

In [ ]:
# # =========
# # OLD
# # =========

# # Set the channelizer to capture even bins
# pc.i.set_channelizer(adc_mode=None,
#                     adc_sampling_mode=None,
#                     adc_bandwidth=None,
#                     adcdaq_mode=None,
#                     data_source='adc',
#                     function=None,
#                     funcgen_left_shift=0,
#                     funcgen_right_shift=0,
#                     # freq_test_bins=None,  # will be passed into function_kwargs
#                     fft_bypass=False,
#                     fft_shift=0b00000000000000,
#                     scaler_bypass=True,
#                     scaler_cap_data_type=4,
#                     scaler_bypass_data_type=None,
#                     postscaler=None,
#                     scaler_eight_bit=None,
#                     scaler_rounding_mode=None,
#                     symmetric_saturation=None,
#                     zero_on_sat=None,
#                     prober_user_flags=None,
#                     offset_binary_encoding=None,
#                     local_sync=True,
#                     channels=list(np.arange(pc.NPOLS)),
#                     gains = None # 1*2**32
#                     )

# # Start data capture for even bins
# pc.i.start_data_capture(period = 0.02, 
#                         source = 'scaler',
#                         mode = pc.ucap.MODE,
#                         channels = list(np.arange(pc.NPOLS)),
#                         data_type = 4, # '2res_even',
#                         select = True
#                         )

# # Capture data!
# t, d_fft_even, c = pc.u_receiver.read_raw_frames(format = '32+32', 
#                                                  ncap = n_cap_fft//pc.n_frames_per_capture(), 
#                                                  split = True,
#                                                  verbose = 0
#                                                  )

# # Set the channelizer to capture odd bins
# pc.i.set_channelizer(adc_mode=None,
#                     adc_sampling_mode=None,
#                     adc_bandwidth=None,
#                     adcdaq_mode=None,
#                     data_source='adc',
#                     function=None,
#                     # funcgen_left_shift=0,
#                     # funcgen_right_shift=0,
#                     # freq_test_bins=None,  # will be passed into function_kwargs
#                     fft_bypass=False,
#                     fft_shift=0b00000000000000,
#                     scaler_bypass=True,
#                     scaler_cap_data_type=5,
#                     scaler_bypass_data_type=None,
#                     postscaler=None,
#                     scaler_eight_bit=None,
#                     scaler_rounding_mode=None,
#                     symmetric_saturation=None,
#                     zero_on_sat=None,
#                     prober_user_flags=None,
#                     offset_binary_encoding=None,
#                     local_sync=True,
#                     channels=list(np.arange(pc.NPOLS)),
#                     gains = None # 1*2**32
#                     )

# # Checks
# pc.i.get_scaler_bypass()
# [pc.i.chan[n].SCALER.CAP_DATA_TYPE for n in range(pc.NPOLS)]

# # Start data capture for even bins
# pc.i.start_data_capture(period = 0.02, 
#                         source = 'scaler',
#                         mode = pc.ucap.MODE,
#                         channels = list(np.arange(pc.NPOLS)),
#                         data_type = 5, # '2res_odd',
#                         select = True
#                         )

# # Capture data!
# t, d_fft_odd, c = pc.u_receiver.read_raw_frames(format = '32+32', 
#                                                  ncap = n_cap_fft//pc.n_frames_per_capture(), 
#                                                  split = True,
#                                                  verbose = 0
#                                                  )

# # Stitch together even and odd bins
# d_fft = np.zeros((d_fft_even.shape[0], d_fft_even.shape[1], 8192), dtype = np.complex128)
# d_fft[..., ::2] = d_fft_even
# d_fft[..., 1::2] = d_fft_odd
# d_fft_mean_psd = np.mean(np.abs(d_fft)**2, axis = 1)

In [ ]:
# Set the channelizer to capture 32+32-bit FFT data
pc.i.set_channelizer(adc_mode=None,
                    adc_sampling_mode=None,
                    adc_bandwidth=None,
                    adcdaq_mode=None,
                    data_source='adc',
                    function=None,
                    funcgen_left_shift=0,
                    funcgen_right_shift=0,
                    # freq_test_bins=None,  # will be passed into function_kwargs
                    fft_bypass=False,
                    fft_shift=0b00000000000000,
                    scaler_bypass=True,
                    scaler_cap_data_type=4,
                    scaler_bypass_data_type=None,
                    postscaler=None,
                    scaler_eight_bit=None,
                    scaler_rounding_mode=None,
                    symmetric_saturation=None,
                    zero_on_sat=None,
                    prober_user_flags=None,
                    offset_binary_encoding=None,
                    local_sync=True,
                    channels=list(np.arange(pc.NPOLS)),
                    gains = None # 1*2**32
                    )

# Start data capture for even bins
pc.i.start_data_capture(period = 0.02, 
                        source = 'scaler',
                        mode = pc.ucap.MODE,
                        channels = list(np.arange(pc.NPOLS)),
                        data_type = 'fft_2x',
                        select = True
                        )

# Capture data!
t, d_fft_32_32_raw, c = pc.u_receiver.read_raw_frames(format = '32+32', 
                                                      ncap = n_cap_fft, # n_cap_fft//pc.n_frames_per_capture(), 
                                                      split = True,
                                                      verbose = 0
                                                      )

# **WARNING!**

# The following line assumes that the `period` as set above ensures 
# that the first frame of any capture is always even, ensuring that 
# even bins are always captured first.

# Stitch together even and odd bins
s = d_fft_32_32_raw.shape
d_fft = d_fft_32_32_raw.reshape(s[0], -1, 2, s[2]).reshape(s[0], -1, s[2]*2, order = 'F')
d_fft_mean_psd = np.mean(np.abs(d_fft)**2, axis = 1)

In [ ]:
pc.i.get_scaler_bypass()

In [ ]:
[pc.i.chan[n].SCALER.CAP_DATA_TYPE for n in range(pc.NPOLS)]

In [ ]:
plt.figure(figsize = (12, 10))
plt.suptitle('raw FFT, 32+32 bits')

for n in range(pc.NPOLS):

    plt.subplot(8, 3, 3*n + 1)
    if n == 0:
        plt.title('real')
    plt.plot(pc.f, d_fft[n, 0, bin_map].real)
    plt.xticks(np.arange(0, 1600 + 400, 400))
    plt.xlim(0, 1600)
    plt.ylabel(f'ADC{n + 1}')
    plt.grid()

    if n + 1 == 8:
        plt.xlabel('frequency [MHz]')
    
    plt.subplot(8, 3, 3*n + 2)
    if n == 0:
        plt.title('imaginary')
    plt.plot(pc.f, d_fft[n, 0, bin_map].imag, color = 'C1')
    plt.xticks(np.arange(0, 1600 + 400, 400))
    plt.xlim(0, 1600)
    plt.grid()

    if n + 1 == 8:
        plt.xlabel('frequency [MHz]')
    
    plt.subplot(8, 3, 3*n + 3)
    if n == 0:
        plt.title('power')
    plt.semilogy(pc.f, np.mean(np.abs(d_fft[n, :, bin_map])**2, axis = 1), color = 'C2')
    plt.xticks(np.arange(0, 1600 + 400, 400))
    plt.xlim(0, 1600)
    plt.grid()

    if n + 1 == 8:
        plt.xlabel('frequency [MHz]')

plt.tight_layout()

In [ ]:
plt.figure(figsize = (12, 4))
plt.title('raw FFT, 32+32 bits, ADC1')
plt.semilogy(pc.f, np.mean(np.abs(d_fft[0, :, bin_map])**2, axis = 1), color = 'C2')
# plt.xlim(275, 350)
plt.ylabel('power [arb.]')
plt.xlabel('frequency [MHz]')
plt.grid()
plt.tight_layout()

In [ ]:
# a = d_fft[0, :, 200].real.flatten().astype(np.int32)
# b = d_fft[0, :, 200].imag.flatten().astype(np.int32)

f_bin = np.arange(1023, 8191 + 1024, 1024)
n_hist_bins = 128
hist_bins = np.arange(n_hist_bins)

plt.figure(figsize = (12, 8))

for n in range(len(f_bin)):

    # c_fft_re, bin_edges = np.histogram(a >> 30, bins = n_bins) # np.arange(-256, 255 + 1)) # , bins = np.arange(-2**20, 2**20 + 1, 1))
    # c_fft_im, bin_edges = np.histogram(b >> 30, bins = n_bins) #np.arange(-256, 255 + 1)) # , bins = np.arange(-2**20, 2**20 + 1, 1))
    c_fft_re, bin_edges = np.histogram(d_fft[0, :, f_bin[n]].real, bins = n_hist_bins)
    c_fft_im, bin_edges = np.histogram(d_fft[0, :, f_bin[n]].imag, bins = n_hist_bins) # np.arange(-256, 255 + 1)) # , bins = np.arange(-2**20, 2**20 + 1, 1))
    # bin_cents = np.arange(-2**20, 2**20)
    # bin_cents = bin_edges
    
    # plt.figure(figsize = (12, 4))
    # plt.suptitle('')
    
    plt.subplot(4, 2, n + 1)
    plt.title(f'{pc.f[f_bin[n]]} MHz')
    plt.plot(hist_bins, c_fft_re/np.sum(c_fft_re))
    plt.plot(hist_bins, c_fft_im/np.sum(c_fft_im))
    # plt.vlines(-8, 0, np.max(c_test_re/np.sum(c_test_re)), color = 'black', linestyle = '--')
    # plt.vlines(8, 0, np.max(c_test_re/np.sum(c_test_re)), color = 'black', linestyle = '--')
    # plt.xticks(bin_cents)

    if n in [0, 2, 4, 6]:
        plt.ylabel('normalized \ncounts')
    if n in [6, 7]:
            plt.xlabel('histogram bin [arb.]')
    plt.grid()
    
plt.tight_layout()

In [ ]:
plt.figure(figsize = (12, 10))
plt.suptitle('raw FFT phase, 32+32 bits')

for n in range(pc.NPOLS):

    plt.subplot(4, 2, n + 1)
    plt.title(f'ADC{n + 1}')
    plt.plot(pc.f, np.angle(np.mean(d_fft[n][:, bin_map], axis = 0)))
    # plt.xticks(np.arange(0, 1600 + 400, 400))
    plt.xlim(0, 100)
    # plt.ylabel(f'ADC{n + 1}')
    plt.grid()

    if n in [6, 7]:
        plt.xlabel('frequency [MHz]')

plt.tight_layout()

In [ ]:
d_fft00 = d_fft[0]*d_fft[0].conj()
d_fft01 = d_fft[0]*d_fft[1].conj()

In [ ]:
d_fft00_mean = np.mean(d_fft00, axis = 0)
d_fft01_mean = np.mean(d_fft01, axis = 0)

In [ ]:
d_fft01.shape

In [ ]:
plt.figure(figsize = (11, 6))
plt.title('correlation phase, raw 32+32-bit FFT data')
plt.plot(pc.f, np.angle(d_fft00_mean)[pc.bin_map], color = 'green', label = 'auto, ADC1')
plt.plot(pc.f, np.angle(d_fft01_mean)[pc.bin_map], color = 'purple', label = 'cross, ADC1-ADC2')
plt.xlim(90, 110)
plt.ylabel('phase [rad.]')
plt.xlabel('frequency [MHz]')
plt.grid()
plt.legend()
plt.tight_layout()

## Compare the 16+16-Bit and 32+32-Bit FFTs

In [ ]:
plt.figure(figsize = (12, 12))
plt.suptitle('FFT bit-depth comparison')

for n in range(pc.NPOLS):

    plt.subplot(8, 2, 2*n + 1)
    if n == 0:
        plt.title('16+16 bits')
    plt.semilogy(pc.f, np.mean(np.abs(d_fft_16_16[n, :, bin_map])**2, axis = 1), color = 'C2')
    plt.xticks(np.arange(0, 1600 + 400, 400))
    plt.ylabel(f'ADC{n + 1}')
    plt.xlim(0, 1600)
    plt.grid()

    if n + 1 == 8:
        plt.xlabel('frequency [MHz]')
    
    plt.subplot(8, 2, 2*n + 2)
    if n == 0:
        plt.title('32+32 bits')
    plt.semilogy(pc.f, np.mean(np.abs(d_fft[n, :, bin_map])**2, axis = 1), color = 'C2')
    plt.xticks(np.arange(0, 1600 + 400, 400))
    plt.xlim(0, 1600)
    plt.grid()

    if n + 1 == 8:
        plt.xlabel('frequency [MHz]')

plt.tight_layout()

In [ ]:
n = 0

plt.figure(figsize = (12, 8))


plt.subplot(211)
plt.title(f'16+16 bits, ADC{n + 1}')
plt.semilogy(pc.f, np.mean(np.abs(d_fft_16_16[n, :, bin_map])**2, axis = 1), color = 'C2')
plt.xticks(np.arange(0, 1600 + 100, 100))
plt.xlim(0, 1600)
plt.ylabel('power [arb.]')
plt.grid()

plt.subplot(212)
plt.title(f'32+32 bits, ADC{n + 1}')
plt.semilogy(pc.f, np.mean(np.abs(d_fft[n, :, bin_map])**2, axis = 1), color = 'C2')
plt.xticks(np.arange(0, 1600 + 100, 100))
plt.xlim(0, 1600)
plt.ylabel('power [arb.]')
plt.xlabel('frequency [MHz]')
plt.grid()

plt.tight_layout()

In [ ]:
for n in range(pc.NPOLS):
    print(n, pc.i.chan[n].FFT.OVERFLOW_COUNT)

## Plot the Raw ADC Data, NumPy FFT of ADC Data, and Full 32+32-Bit Firmware FFT

In [ ]:
plt.figure(figsize = (16, 16))
plt.suptitle('numpy and firmware FFT comparison')

for n in range(pc.NPOLS):

    plt.subplot(8, 3, 3*n + 1)
    if n == 0:
        plt.title('ADC data')
    plt.plot(np.arange(2**14), d_adc[n, 0]) #, marker = '.', s = 25)
    plt.hlines(8191, 0, 128, color = 'black', linestyle = '--')
    plt.hlines(-8192, 0, 128, color = 'black', linestyle = '--')
    plt.ylim(-256, 255)
    # plt.yticks([-8192, -4096, -2048, 2047, 4095, 8181])
    plt.yticks([-256, -128, -64, 63, 127, 255])
    plt.xlim(0, 256)
    plt.ylabel(f'ADC{n + 1}')
    plt.xlabel('sample')
    plt.grid()
    
    if n + 1 == 8:
        plt.xlabel('sample')
        
    plt.subplot(8, 3, 3*n + 2)
    if n == 0:
        plt.title('numpy FFT of ADC data')
    plt.semilogy(pc.f, d_adc_psd[n][:-1], color = 'C1')
    plt.xticks(np.arange(0, 1600 + 400, 400))
    plt.xlim(0, 1600)
    plt.grid()

    if n + 1 == 8:
        plt.xlabel('frequency [MHz]')
    
    plt.subplot(8, 3, 3*n + 3)
    if n == 0:
        plt.title('firmware FFT, 32+32 bits')
    plt.semilogy(pc.f, d_fft_mean_psd[n, bin_map], color = 'C2', label = 'firmware fft')
    plt.xticks(np.arange(0, 1600 + 400, 400))
    plt.xlim(0, 1600)
    plt.grid()

    if n + 1 == 8:
        plt.xlabel('frequency [MHz]')

plt.tight_layout()

# Compute Floating-Point Digital Gains

With the 32+32-bit FFT, the digital gains now multiply the FFT output to a 49+49-bit number, of which the upper 4+4 bits are preserved.

Also recall that the digital gains now support floating-point format, with 11 bits dedicated to the linear component and 5 bits dedicated to the log component per frequency bin. Also note that there is a "common" log component that applies to each channelizer, and is an additional 5-bit value that is *added* to the per-bin log component, for a total of 6 bits for the log component. Therefore, the maximum digital gain is $2^{11} \times 2^{2^6} = 2048 \times 2^{64}$. Each frequency bin now has a unique digital gain (i.e., the log component is no longer shared for a given analog channel).

## Test with FUNCGEN, Flat Spectrum

I will first test the new gains with a flat spectrum, to confirm that the values are found in the right place after being read from the Scaler.

By setting set_data_source('a', a = 1), I am generating a FUNCGEN signal whose value is 1 for a single frame, $M = 2^{14}$. The FUNCGEN generates 14-bit values, which are shifted by 18 bits when the FFT is bypassed. Even (odd) indices of the FUNCGEN become the real (imaginary) parts of the 32+32-bit output of the FFT when it is bypassed. 

Therefore, the configuration below, when interpreted as 32+32-bit FFT data, will be equivalent to $2^{18} + j~2^{18}$ for all frequency bins.

Another important subtelty to keep in mind is that the frequency bins of the channelizers are scrambled, such that the inverse bin map must be applied to any custom function written to the buffers of the FUNCGEN. However, applying the inverse bin map is not strictly necessary given that the data is captured in scrambled frequency bin-order. Therefore, if a custom function is written in a certain order in Python, it will be read back in the same order, as well as any downstream data products. 

As such, I will not apply a bin map anywhere in the "flat" section for simplicity. However, with frequency-dependent or real data, we will need to apply a bin map to make sense of the data.

In [ ]:
pc.i.set_channelizer(data_source='a',
                     a = 1,
                     funcgen_left_shift = 0,
                     funcgen_right_shift = 0,
                     fft_bypass=None,
                     fft_shift=None,
                     scaler_bypass=None,
                     channels=list(np.arange(pc.NPOLS)),
                     gains=None,
                     )

# pc.i.set_channelizer(
#             adc_mode=None,
#             adc_sampling_mode=None,
#             adc_bandwidth=None,
#             adcdaq_mode=None,
#             data_source='a',
#             a = 1,
#             function=None,
#             # funcgen_left_shift=None,
#             # funcgen_right_shift=None,
#             # freq_test_bins=None,  # will be passed into function_kwargs
#             fft_bypass=None,
#             fft_shift=None,
#             scaler_bypass=None,
#             scaler_cap_data_type=None,
#             scaler_bypass_data_type=None,
#             postscaler=None,
#             scaler_eight_bit=None,
#             scaler_rounding_mode=None,
#             symmetric_saturation=None,
#             zero_on_sat=None,
#             prober_user_flags=None,
#             offset_binary_encoding=None,
#             local_sync=True,
#             channels=list(np.arange(pc.NPOLS)),
#             gains=None,
#             )

# Start data capture, grabbing data at the output of the
# scaler, which has been bypassed.
pc.i.start_data_capture(period = 0.02, 
                        source = 'funcgen', # 'adc',
                        mode = pc.ucap.MODE,
                        channels = list(np.arange(pc.NPOLS)),
                        select = True,
                        data_type = None
                        )

# Capture data!
t, d_adc_flat, c = pc.u_receiver.read_raw_frames(format = '16', 
                                                 ncap = 1,
                                                 split = True, 
                                                 verbose = 0
                                                 )

d_adc_flat = d_adc_flat >> 2 # shift to 14 bits

In [ ]:
plt.figure(figsize = (12, 4))
plt.plot(d_adc_flat[0, 0])
plt.grid()
plt.tight_layout()

In [ ]:
# =========
# OLD
# =========

# # Check to make sure we're configured to receive channelizer data, not correlated data:
# if pc.ucap.OUTPUT_SOURCE_SEL != 0:
#     pc.ucap.OUTPUT_SOURCE_SEL = 0

# # Make sure we can get data from all channels:
# if pc.ucap.MODE != 0:
#     pc.ucap.MODE = 0 

# # Bypass the FFT and the SCALER first. Set the data source as desired
# for n in range(pc.NPOLS):
#     pc.i.chan[n].FUNCGEN.set_data_source('a', a = 1)
#     pc.i.chan[n].FFT.BYPASS = 1
    
#     # if not pc.i.chan[n].SCALER.BYPASS:
#     pc.i.chan[n].SCALER.BYPASS = 1
#     # pc.i.chan[n].SCALER.DATA_TYPE = 2 
#     pc.i.chan[n].SCALER.CAP_DATA_TYPE = 4 # even bins of raw 32+32-bit FFT

# # Start data capture, grabbing data at the output of the
# # scaler, which has been bypassed.
# pc.i.start_data_capture(period = 0.02, 
#                          source = 'scaler'
#                          )

# # Capture data!
# t, d_fft_flat_even, c = pc.u_receiver.read_raw_frames(format = '32+32', 
#                                                        ncap = 1, 
#                                                        split = True,
#                                                        verbose = 0
#                                                        )

# # ====
# # Now do the same for the odd bins, switching Scaler modes:

# # Bypass the FFT and the SCALER first. Set the data source as desired
# for n in range(pc.NPOLS):
#     # pc.i.chan[n].SCALER.DATA_TYPE = 3
#     pc.i.chan[n].SCALER.CAP_DATA_TYPE = 5 # odd bins of raw 32+32-bit FFT

# # Start data capture, grabbing data at the output of the
# # scaler, which has been bypassed.
# pc.i.start_data_capture(period = 0.02, 
#                          source = 'scaler'
#                          )

# # Capture data!
# t, d_fft_flat_odd, c = pc.u_receiver.read_raw_frames(format = '32+32', 
#                                                       ncap = 1, 
#                                                       split = True,
#                                                       verbose = 0
#                                                       )

# d_fft_flat = np.zeros((d_fft_flat_even.shape[0], d_fft_flat_even.shape[1], 8192), dtype = np.complex128)
# d_fft_flat[..., ::2] = d_fft_flat_even
# d_fft_flat[..., 1::2] = d_fft_flat_odd
# d_fft_flat_mean_psd = np.mean(np.abs(d_fft_flat)**2, axis = 1)

In [ ]:
# =========
# OLD
# =========

# # Set the channelizer to capture even bins
# pc.i.set_channelizer(adc_mode=None,
#                      adc_sampling_mode=None,
#                      adc_bandwidth=None,
#                      adcdaq_mode=None,
#                      data_source='a',
#                      a = 1,
#                      function=None,
#                      # funcgen_left_shift=0,
#                      # funcgen_right_shift=0,
#                      # freq_test_bins=None,  # will be passed into function_kwargs
#                      fft_bypass=True,
#                      fft_shift=0b00000000000000,
#                      scaler_bypass=True,
#                      scaler_cap_data_type=4,
#                      scaler_bypass_data_type=None,
#                      postscaler=None,
#                      scaler_eight_bit=None,
#                      scaler_rounding_mode=None,
#                      symmetric_saturation=None,
#                      zero_on_sat=None,
#                      prober_user_flags=None,
#                      offset_binary_encoding=None,
#                      local_sync=True,
#                      channels=list(np.arange(pc.NPOLS)),
#                      gains = None # 1*2**32
#                      )


# # Start data capture for even bins
# pc.i.start_data_capture(period = 0.02, 
#                         source = 'scaler',
#                         mode = pc.ucap.MODE,
#                         channels = list(np.arange(pc.NPOLS)),
#                         data_type = 4, # '2res_even',
#                         select = True
#                         )

# # Capture data!
# t, d_fft_flat_even, c = pc.u_receiver.read_raw_frames(format = '32+32', 
#                                                        ncap = 1, 
#                                                        split = True,
#                                                        verbose = 0
#                                                        )

# # Set the channelizer to capture odd bins
# pc.i.set_channelizer(adc_mode=None,
#                      adc_sampling_mode=None,
#                      adc_bandwidth=None,
#                      adcdaq_mode=None,
#                      data_source='a',
#                      a = 1,
#                      function=None,
#                      # funcgen_left_shift=0,
#                      # funcgen_right_shift=0,
#                      # freq_test_bins=None,  # will be passed into function_kwargs
#                      fft_bypass=True,
#                      fft_shift=0b00000000000000,
#                      scaler_bypass=True,
#                      scaler_cap_data_type=5,
#                      scaler_bypass_data_type=None,
#                      postscaler=None,
#                      scaler_eight_bit=None,
#                      scaler_rounding_mode=None,
#                      symmetric_saturation=None,
#                      zero_on_sat=None,
#                      prober_user_flags=None,
#                      offset_binary_encoding=None,
#                      local_sync=True,
#                      channels=list(np.arange(pc.NPOLS)),
#                      gains = None 
#                      )

# # Checks
# pc.i.get_scaler_bypass()
# [pc.i.chan[n].SCALER.CAP_DATA_TYPE for n in range(pc.NPOLS)]

# # Start data capture for even bins
# pc.i.start_data_capture(period = 0.02, 
#                         source = 'scaler',
#                         mode = pc.ucap.MODE,
#                         channels = list(np.arange(pc.NPOLS)),
#                         data_type = 5, # '2res_odd',
#                         select = True
#                         )

# # Capture data!
# t, d_fft_flat_odd, c = pc.u_receiver.read_raw_frames(format = '32+32', 
#                                                       ncap = 1, 
#                                                       split = True,
#                                                       verbose = 0
#                                                       )

# d_fft_flat = np.zeros((d_fft_flat_even.shape[0], d_fft_flat_even.shape[1], 8192), dtype = np.complex128)
# d_fft_flat[..., ::2] = d_fft_flat_even
# d_fft_flat[..., 1::2] = d_fft_flat_odd
# d_fft_flat_mean_psd = np.mean(np.abs(d_fft_flat)**2, axis = 1)

In [ ]:
# Set the channelizer to capture 32+32-bit FFT data
pc.i.set_channelizer(adc_mode=None,
                     adc_sampling_mode=None,
                     adc_bandwidth=None,
                     adcdaq_mode=None,
                     data_source='a',
                     a = 1,
                     function=None,
                     # funcgen_left_shift=0,
                     # funcgen_right_shift=0,
                     # freq_test_bins=None,  # will be passed into function_kwargs
                     fft_bypass=True,
                     fft_shift=0b00000000000000,
                     scaler_bypass=True,
                     scaler_cap_data_type=4,
                     scaler_bypass_data_type=None,
                     postscaler=None,
                     scaler_eight_bit=None,
                     scaler_rounding_mode=None,
                     symmetric_saturation=None,
                     zero_on_sat=None,
                     prober_user_flags=None,
                     offset_binary_encoding=None,
                     local_sync=True,
                     channels=list(np.arange(pc.NPOLS)),
                     gains = None # 1*2**32
                     )

# Start data capture
pc.i.start_data_capture(period = 0.02, 
                        source = 'scaler',
                        mode = pc.ucap.MODE,
                        channels = list(np.arange(pc.NPOLS)),
                        data_type = 'fft_2x',
                        select = True
                        )

# Capture data!
t, d_fft_flat_raw, c = pc.u_receiver.read_raw_frames(format = '32+32', 
                                                     ncap = n_cap_fft, # n_cap_fft//pc.n_frames_per_capture(), 
                                                     split = True,
                                                     verbose = 0
                                                     )

# **WARNING!**

# The following line assumes that the `period` as set above ensures 
# that the first frame of any capture is always even, ensuring that 
# even bins are always captured first.

# Stitch together even and odd bins
s = d_fft_flat_raw.shape
d_fft_flat = d_fft_flat_raw.reshape(s[0], -1, 2, s[2]).reshape(s[0], -1, s[2]*2, order = 'F')
d_fft_flat_mean_mag = np.mean(np.abs(d_fft_flat), axis = 1)

In [ ]:
d_fft_flat.shape

In [ ]:
d_fft_flat[0,0,0]

In [ ]:
np.log2(262144)

In [ ]:
np.abs(d_fft_flat[0,0,0])

In [ ]:
np.sqrt(2*(2**18)**2)

In [ ]:
# Plot the flat data to verify that it's being set up and read correctly

plt.figure(figsize = (12, 6))

plt.subplot(311)
plt.title('test flat FFT data')
plt.plot(pc.f, d_fft_flat[0, 0, bin_map].real, label = 'measured')
plt.hlines(2**18, 0, 1600, color = 'C1', linestyle = '--', label = 'expected')
plt.ylabel('real')
plt.xlim(0, 1600)
plt.grid()
plt.legend()

plt.subplot(312)
plt.plot(pc.f, d_fft_flat[0, 0, bin_map].imag)
plt.hlines(2**18, 0, 1600, color = 'C1', linestyle = '--')
plt.ylabel('imag')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(313)
plt.plot(pc.f, d_fft_flat_mean_mag[0, bin_map])
plt.hlines(np.sqrt(2*(2**18)**2), 0, 1600, color = 'C1', linestyle = '--')
plt.ylabel('magnitude')
plt.xlabel('frequency [MHz]')
plt.xlim(0, 1600)
plt.grid()

plt.tight_layout()

### Set the Test Gains and Capture Scaler Data

Because we have set a single integer value as our signal, it is impossible for us to recover any value from the Scaler that is not an integer! There is no noise on the signal, so its output is deterministic. 

Therefore, the target magnitude must be set to be the quadrature sum of two integers For example, if I want the real and imaginary components to always be +1 (positive since we have set the spectrum to always be positive), then my Scaler magnitude must be

\begin{equation}
    |\text{Scaler}| = \sqrt{1^2 + 1^2} = \sqrt{2}.
\end{equation}

I earlier had the real and imaginary components as always +2, but this ends up saturating the correlator later on...

In [ ]:
test_target = np.sqrt(2) # The target magnitude as a 4-bit number
test_target_full = test_target*2**44 # The target magnitude as a 4-bit number on top of a 44-bit number 

In [ ]:
test_gains_flat = test_target_full/d_fft_flat_mean_mag

In [ ]:
plt.figure(figsize = (12, 6))

plt.subplot(311)
plt.title('verification of gain computation')
plt.plot(pc.f, d_fft_flat_mean_mag[0, bin_map])
plt.ylabel('FFT magnitude')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(312)
plt.plot(pc.f, test_gains_flat[0, bin_map])
plt.ylabel('gains')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(313)
plt.plot(pc.f, np.mean(np.abs(d_fft_flat[0]*test_gains_flat[0])/2**44, axis = 0)[bin_map], label = 'gain $\\times$ |FFT|')
plt.hlines(test_target, 0, 1600, color = 'red', linestyle = '--', label = 'target')
plt.ylabel('simulated \nscaler magnitude')
plt.xlabel('frequency [MHz]')
plt.xlim(0, 1600)
plt.grid()
plt.legend()

plt.tight_layout()

### Set the Test Digital Gains

In [ ]:
pc.i.chan[0].SCALER.USE_COMPLEX_GAINS

In [ ]:
pc.i.chan[0].SCALER.USE_FLOAT_GAINS = 1

In [ ]:
pc.i.chan[0].SCALER.set_gain_table(test_gains_flat[0], bank=0, gain_timestamp=None, log_gain=None)

### Check the upper 16+16 bits of the 48+48-bit value after applying digital gains to the FFT data

In [ ]:
# =========
# OLD
# =========

# # Check to make sure we're configured to receive channelizer data, not correlated data:
# if pc.ucap.OUTPUT_SOURCE_SEL != 0:
#     pc.ucap.OUTPUT_SOURCE_SEL = 0

# # Make sure we can get data from all channels:
# if pc.ucap.MODE != 0:
#     pc.ucap.MODE = 0 

# # Check that the FFT is not bypassed, the SCALER is bypassed, and set the data source as desired
# for n in range(pc.NPOLS):
#     pc.i.chan[n].FUNCGEN.set_data_source('a', a = 1)

#     pc.i.chan[n].SCALER.BYPASS = 1
#     # pc.i.chan[n].SCALER.DATA_TYPE = 1 # Send the most significant 16+16 bits of the saturated post-gain FFT values
#     pc.i.chan[n].SCALER.CAP_DATA_TYPE = 2 # MSB of post-gain FFT value with saturation

# # Start data capture, grabbing data at the output of the
# # scaler, which has been bypassed.
# pc.i.start_data_capture(period = 0.02, 
#                          source = 'scaler'
#                          )

# # Capture data!
# t, d_post_gain_flat, c = pc.u_receiver.read_raw_frames(format = '16+16', 
#                                                        ncap = 1, 
#                                                        split = True,
#                                                        verbose = 0
#                                                        )

In [ ]:
# Set the channelizer to capture post-gain data
pc.i.set_channelizer(adc_mode=None,
                     adc_sampling_mode=None,
                     adc_bandwidth=None,
                     adcdaq_mode=None,
                     data_source='a',
                     a = 1,
                     function=None,
                     funcgen_left_shift=0,
                     funcgen_right_shift=0,
                     # freq_test_bins=None,  # will be passed into function_kwargs
                     fft_bypass=True,
                     fft_shift=0b00000000000000,
                     scaler_bypass=True,
                     scaler_cap_data_type=2,
                     scaler_bypass_data_type=None,
                     postscaler=None,
                     scaler_eight_bit=None,
                     scaler_rounding_mode=None,
                     symmetric_saturation=None,
                     zero_on_sat=None,
                     prober_user_flags=None,
                     offset_binary_encoding=None,
                     local_sync=True,
                     channels=list(np.arange(pc.NPOLS)),
                     gains = None # 1*2**32
                     )


# Start data capture, grabbing data at the output of the
# scaler, which has been bypassed.
pc.i.start_data_capture(period = 0.02, 
                        source = 'scaler',
                        mode = pc.ucap.MODE,
                        channels = list(np.arange(pc.NPOLS)),
                        data_type = 'fft_gain',
                        select = True
                        )

# Capture data!
t, d_post_gain_flat, c = pc.u_receiver.read_raw_frames(format = '16+16', 
                                                       ncap = 1, 
                                                       split = True,
                                                       verbose = 0
                                                       )

In [ ]:
d_post_gain_flat.shape

In [ ]:
plt.figure(figsize = (12, 6))

plt.subplot(311)
plt.title('post-gain upper 16+16 bits, flat')
plt.plot(pc.f, d_post_gain_flat[0, 0, bin_map].real)
plt.ylabel('real')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(312)
plt.plot(pc.f, d_post_gain_flat[0, 0, bin_map].imag)
plt.ylabel('imag')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(313)
plt.plot(pc.f, np.abs(d_post_gain_flat[0, 0, bin_map]), label = 'measured')
plt.hlines(test_target*2**12, 0, 1600, color = 'C1', linestyle = '--', label = 'expected')
plt.ylabel('magnitude')
plt.xlabel('frequency [MHz]')
plt.xlim(0, 1600)
plt.grid()
plt.legend()

plt.tight_layout()

### Capture Scaler Data

In [ ]:
# =========
# OLD
# =========

# # Check to make sure we're configured to receive channelizer data, not correlated data:
# if pc.ucap.OUTPUT_SOURCE_SEL != 0:
#     pc.ucap.OUTPUT_SOURCE_SEL = 0

# # Make sure we can get data from all channels:
# if pc.ucap.MODE != 0:
#     pc.ucap.MODE = 0 

# # Check that the FFT is bypassed, the SCALER is not bypassed, and set the data source as desired
# for n in range(pc.NPOLS):
#     pc.i.chan[n].FUNCGEN.set_data_source('a', a = 1)
#     # pc.i.chan[n].FUNCGEN.set_data_source('ab', a = 1, b = -1)

#     # if not pc.i.chan[n].SCALER.BYPASS:
#     pc.i.chan[n].SCALER.BYPASS = 0
#     # pc.i.chan[n].SCALER.DATA_TYPE = 0 # Send 4-bit scaler values
#     pc.i.chan[n].SCALER.CAP_DATA_TYPE = 0 # Main scaler output

# # Start data capture, grabbing data at the output of the
# # scaler, which has been bypassed.
# pc.i.start_data_capture(period = 0.02, 
#                          source = 'scaler'
#                          )

# t, d_sclr_test, c = pc.u_receiver.read_raw_frames(format = '16+16', 
#                                                    ncap = 1, 
#                                                    split = True,
#                                                    verbose = 0
#                                                    )

# d_sclr_test = d_sclr_test / 2**12

In [ ]:
# Set the channelizer to capture Scaler data
pc.i.set_channelizer(adc_mode=None,
                     adc_sampling_mode=None,
                     adc_bandwidth=None,
                     adcdaq_mode=None,
                     data_source='a',
                     a = 1,
                     function=None,
                     # funcgen_left_shift=0,
                     # funcgen_right_shift=0,
                     # freq_test_bins=None,  # will be passed into function_kwargs
                     fft_bypass=True,
                     fft_shift=0b00000000000000,
                     scaler_bypass=False,
                     scaler_cap_data_type=0,
                     scaler_bypass_data_type=None,
                     postscaler=None,
                     scaler_eight_bit=None,
                     scaler_rounding_mode=None,
                     symmetric_saturation=None,
                     zero_on_sat=None,
                     prober_user_flags=None,
                     offset_binary_encoding=None,
                     local_sync=True,
                     channels=list(np.arange(pc.NPOLS)),
                     gains = None # 1*2**32
                     )


# Start data capture for 4+4-bit Scaler data
pc.i.start_data_capture(period = 0.02, 
                        source = 'scaler',
                        mode = pc.ucap.MODE,
                        channels = list(np.arange(pc.NPOLS)),
                        data_type = 'fft_4bit',
                        select = True
                        )

# Capture data!
t, d_sclr_test, c = pc.u_receiver.read_raw_frames(format = '16+16', 
                                                  ncap = 1, 
                                                  split = True,
                                                  verbose = 0
                                                  )

d_sclr_test = d_sclr_test / 2**12

In [ ]:
d_sclr_test.shape

In [ ]:
d_sclr_test_mean_mag = np.mean(np.abs(d_sclr_test[0]), axis = 0)
d_sclr_test_mean_mag

### Plot the Scaler Magnitude and Histograms

In [ ]:
plt.figure(figsize = (12, 4))
plt.plot(pc.f, d_sclr_test_mean_mag[bin_map], label = 'mean')
plt.hlines(test_target, 0, 1600, color = 'red', linestyle = '--', label = 'target')
plt.ylabel('scaler magnitude')
plt.xlabel('frequency [MHz]')
plt.xlim(0, 1600)
plt.legend()
plt.grid()
plt.tight_layout()

In [ ]:
c_test_re, bin_edges = np.histogram(d_sclr_test[0].real.flatten(), bins = np.arange(-7.5, 7.5 + 1, 1))
c_test_im, bin_edges = np.histogram(d_sclr_test[0].imag.flatten(), bins = np.arange(-7.5, 7.5 + 1, 1))
bin_cents = np.arange(-7, 7 + 1)

plt.figure(figsize = (12, 4))

plt.subplot(121)
plt.title('real')
plt.bar(bin_cents, c_test_re/np.sum(c_test_re))
plt.vlines(-8, 0, np.max(c_test_re/np.sum(c_test_re)), color = 'black', linestyle = '--')
plt.vlines(8, 0, np.max(c_test_re/np.sum(c_test_re)), color = 'black', linestyle = '--')
plt.scatter(1, 1, color = 'red', label = 'expected')
plt.xticks(bin_cents)
plt.ylabel('normalized counts')
plt.xlabel('scaler LSBs')
plt.grid()
plt.legend()

plt.subplot(122)
plt.title('imag')
plt.bar(bin_cents, c_test_im/np.sum(c_test_im))
plt.vlines(-8, 0, np.max(c_test_im/np.sum(c_test_im)), color = 'black', linestyle = '--')
plt.vlines(8, 0, np.max(c_test_im/np.sum(c_test_im)), color = 'black', linestyle = '--')
plt.scatter(1, 1, color = 'red', label = 'expected')
plt.xticks(bin_cents)
plt.xlabel('scaler LSBs')
plt.grid()

plt.tight_layout()

## Test with Ramp Function

As with the flat function, the ramp function will have a deterministic output. Therefore, we can use the same target as with the flat function; however, we will now have a gain that varies in frequency.

In [ ]:
ramp_half = np.arange(pc.M//2)
ramp = np.zeros(len(ramp_half)*2)
ramp[::2] = ramp_half[inv_bin_map]
ramp[1::2] = -1*ramp_half[inv_bin_map]

In [ ]:
ramp.shape

In [ ]:
plt.figure(figsize = (12, 6))
plt.subplot(211)
plt.title('simulated custom ramp function, interpreted as an FFT')
plt.plot(pc.f, ramp[::2][bin_map])
plt.xlim(0, 1600)
plt.ylabel('real')
plt.grid()
plt.subplot(212)
plt.plot(pc.f, ramp[1::2][bin_map])
plt.xlim(0, 1600)
plt.ylabel('imag')
plt.xlabel('frequency [MHz]')
plt.grid()
plt.tight_layout()

In [ ]:
# =========
# OLD
# =========

# # Check to make sure we're configured to receive channelizer data, not correlated data:
# if pc.ucap.OUTPUT_SOURCE_SEL != 0:
#     pc.ucap.OUTPUT_SOURCE_SEL = 0

# # Make sure we can get data from all channels:
# if pc.ucap.MODE != 0:
#     pc.ucap.MODE = 0 

# # Bypass the FFT and the SCALER first. Set the data source as desired
# for n in range(pc.NPOLS):
#     # pc.i.chan[n].FUNCGEN.set_data_source('a', a = 1)
#     # pc.i.chan[n].FUNCGEN.set_data_source('buffer', steps)
#     # pc.i.chan[n].FUNCGEN.set_data_source('sin')
#     pc.i.chan[n].FUNCGEN.set_data_source('buffer', ramp)
#     pc.i.chan[n].FFT.BYPASS = 1
    
#     # if not pc.i.chan[n].SCALER.BYPASS:
#     pc.i.chan[n].SCALER.BYPASS = 1
#     # pc.i.chan[n].SCALER.DATA_TYPE = 2 
#     pc.i.chan[n].SCALER.CAP_DATA_TYPE = 4 # even bins of raw 32+32-bit FFT

# # Start data capture, grabbing data at the output of the
# # scaler, which has been bypassed.
# pc.i.start_data_capture(period = 0.02, 
#                          source = 'scaler'
#                          )

# # Capture data!
# t, d_fft_ramp_even, c = pc.u_receiver.read_raw_frames(format = '32+32', 
#                                                        ncap = 1, 
#                                                        split = True,
#                                                        verbose = 0
#                                                        )

# # ====
# # Now do the same for the odd bins, switching Scaler modes:

# # Bypass the FFT and the SCALER first. Set the data source as desired
# for n in range(pc.NPOLS):
#     # pc.i.chan[n].SCALER.DATA_TYPE = 3
#     pc.i.chan[n].SCALER.CAP_DATA_TYPE = 5 # odd bins of raw 32+32-bit FFT

# # Start data capture, grabbing data at the output of the
# # scaler, which has been bypassed.
# pc.i.start_data_capture(period = 0.02, 
#                          source = 'scaler'
#                          )

# # Capture data!
# t, d_fft_ramp_odd, c = pc.u_receiver.read_raw_frames(format = '32+32', 
#                                                       ncap = 1, 
#                                                       split = True,
#                                                       verbose = 0
#                                                       )

# d_fft_ramp = np.zeros((d_fft_ramp_even.shape[0], d_fft_ramp_even.shape[1], 8192), dtype = np.complex128)
# d_fft_ramp[..., ::2] = d_fft_ramp_even
# d_fft_ramp[..., 1::2] = d_fft_ramp_odd
# d_fft_ramp_mean_mag = np.mean(np.abs(d_fft_ramp), axis = 1)

In [ ]:
# =========
# OLD
# =========

# # Set the channelizer to capture even bins
# pc.i.set_channelizer(adc_mode=None,
#                     adc_sampling_mode=None,
#                     adc_bandwidth=None,
#                     adcdaq_mode=None,
#                     data_source='buffer',
#                     buffer = ramp,
#                     function=None,
#                     funcgen_left_shift=0,
#                     funcgen_right_shift=0,
#                     # freq_test_bins=None,  # will be passed into function_kwargs
#                     fft_bypass=True,
#                     fft_shift=0b00000000000000,
#                     scaler_bypass=True,
#                     scaler_cap_data_type=4,
#                     scaler_bypass_data_type=None,
#                     postscaler=None,
#                     scaler_eight_bit=None,
#                     scaler_rounding_mode=None,
#                     symmetric_saturation=None,
#                     zero_on_sat=None,
#                     prober_user_flags=None,
#                     offset_binary_encoding=None,
#                     local_sync=True,
#                     channels=list(np.arange(pc.NPOLS)),
#                     gains = None # 1*2**32
#                     )

# # Start data capture for even bins
# pc.i.start_data_capture(period = 0.02, 
#                         source = 'scaler',
#                         mode = pc.ucap.MODE,
#                         channels = list(np.arange(pc.NPOLS)),
#                         data_type = 4, # '2res_even',
#                         select = True
#                         )

# # Capture data!
# t, d_fft_ramp_even, c = pc.u_receiver.read_raw_frames(format = '32+32', 
#                                                       ncap = n_cap_fft//pc.n_frames_per_capture(), 
#                                                       split = True,
#                                                       verbose = 0
#                                                       )

# # Set the channelizer to capture odd bins
# pc.i.set_channelizer(adc_mode=None,
#                     adc_sampling_mode=None,
#                     adc_bandwidth=None,
#                     adcdaq_mode=None,
#                     data_source='buffer',
#                     buffer = ramp,
#                     function=None,
#                     # funcgen_left_shift=0,
#                     # funcgen_right_shift=0,
#                     # freq_test_bins=None,  # will be passed into function_kwargs
#                     fft_bypass=True,
#                     fft_shift=0b00000000000000,
#                     scaler_bypass=True,
#                     scaler_cap_data_type=5,
#                     scaler_bypass_data_type=None,
#                     postscaler=None,
#                     scaler_eight_bit=None,
#                     scaler_rounding_mode=None,
#                     symmetric_saturation=None,
#                     zero_on_sat=None,
#                     prober_user_flags=None,
#                     offset_binary_encoding=None,
#                     local_sync=True,
#                     channels=list(np.arange(pc.NPOLS)),
#                     gains = None # 1*2**32
#                     )

# # Start data capture for even bins
# pc.i.start_data_capture(period = 0.02, 
#                         source = 'scaler',
#                         mode = pc.ucap.MODE,
#                         channels = list(np.arange(pc.NPOLS)),
#                         data_type = 5, # '2res_odd',
#                         select = True
#                         )

# # Capture data!
# t, d_fft_ramp_odd, c = pc.u_receiver.read_raw_frames(format = '32+32', 
#                                                      ncap = n_cap_fft//pc.n_frames_per_capture(), 
#                                                      split = True,
#                                                      verbose = 0
#                                                      )

# d_fft_ramp = np.zeros((d_fft_ramp_even.shape[0], d_fft_ramp_even.shape[1], 8192), dtype = np.complex128)
# d_fft_ramp[..., ::2] = d_fft_ramp_even
# d_fft_ramp[..., 1::2] = d_fft_ramp_odd
# d_fft_ramp_mean_mag = np.mean(np.abs(d_fft_ramp), axis = 1)

In [ ]:
# Set the channelizer to capture odd bins
pc.i.set_channelizer(adc_mode=None,
                    adc_sampling_mode=None,
                    adc_bandwidth=None,
                    adcdaq_mode=None,
                    data_source='buffer',
                    buffer = ramp,
                    function=None,
                    # funcgen_left_shift=0,
                    # funcgen_right_shift=0,
                    # freq_test_bins=None,  # will be passed into function_kwargs
                    fft_bypass=True,
                    fft_shift=0b00000000000000,
                    scaler_bypass=True,
                    scaler_cap_data_type=5,
                    scaler_bypass_data_type=None,
                    postscaler=None,
                    scaler_eight_bit=None,
                    scaler_rounding_mode=None,
                    symmetric_saturation=None,
                    zero_on_sat=None,
                    prober_user_flags=None,
                    offset_binary_encoding=None,
                    local_sync=True,
                    channels=list(np.arange(pc.NPOLS)),
                    gains = None # 1*2**32
                    )

# Start data capture
pc.i.start_data_capture(period = 0.02, 
                        source = 'scaler',
                        mode = pc.ucap.MODE,
                        channels = list(np.arange(pc.NPOLS)),
                        data_type = 'fft_2x',
                        select = True
                        )

# Capture data!
t, d_fft_ramp_raw, c = pc.u_receiver.read_raw_frames(format = '32+32', 
                                                     ncap = n_cap_fft, # n_cap_fft//pc.n_frames_per_capture(), 
                                                     split = True,
                                                     verbose = 0
                                                     )

# **WARNING!**

# The following line assumes that the `period` as set above ensures 
# that the first frame of any capture is always even, ensuring that 
# even bins are always captured first.

# Stitch together even and odd bins
s = d_fft_ramp_raw.shape
d_fft_ramp = d_fft_ramp_raw.reshape(s[0], -1, 2, s[2]).reshape(s[0], -1, s[2]*2, order = 'F')
d_fft_ramp_mean_mag = np.mean(np.abs(d_fft_ramp), axis = 1)

In [ ]:
# Plot the flat data to verify that it's being set up and read correctly

plt.figure(figsize = (12, 8))

plt.subplot(311)
plt.title('test ramp spectrum')
plt.plot(pc.f, d_fft_ramp[0, 0, bin_map].real, label = 'measured')
plt.plot(pc.f, ramp[::2][bin_map]*2**18, color = 'C1', linestyle = '--', label = 'expected')
plt.ylabel('real')
plt.xlim(0, 1600)
plt.grid()
plt.legend()

plt.subplot(312)
plt.plot(pc.f, d_fft_ramp[0, 0, bin_map].imag)
plt.plot(pc.f, ramp[1::2][bin_map]*2**18, color = 'C1', linestyle = '--', label = 'expected')
plt.ylabel('imag')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(313)
plt.plot(pc.f, d_fft_ramp_mean_mag[0][bin_map])
plt.plot(pc.f, np.sqrt((ramp[::2][bin_map]**2 + ramp[1::2][bin_map]**2))*(2**18), color = 'C1', linestyle = '--', label = 'expected')
plt.ylabel('magnitude')
plt.xlabel('frequency [MHz]')
plt.xlim(0, 1600)
plt.grid()

plt.tight_layout()

### Set the Test Gains for the Ramp Function

In [ ]:
test_gains_ramp = test_target_full/d_fft_ramp_mean_mag

In [ ]:
d_fft_ramp_mean_mag.shape

In [ ]:
plt.figure(figsize = (12, 8))

plt.subplot(311)
plt.title('verification of gain computation, computed gains')
plt.plot(pc.f, d_fft_ramp_mean_mag[0, bin_map])
plt.ylabel('|FFT|')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(312)
plt.semilogy(pc.f, test_gains_ramp[0, bin_map])
plt.ylabel('gains')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(313)
plt.plot(pc.f[1:], d_fft_ramp_mean_mag[0, bin_map][1:]*test_gains_ramp[0, bin_map][1:]/2**44, label = 'gain $\\times$ |FFT|')
plt.hlines(test_target, 0, 1600, color = 'red', linestyle = '--', label = 'target')
plt.ylabel('simulated |scaler|')
plt.xlabel('frequency [MHz]')
plt.xlim(0, 1600)
plt.grid()
plt.legend()

plt.tight_layout()

In [ ]:
# Treat infinities
test_gains_ramp[0][np.where(test_gains_ramp[0] == np.inf)[0]] = 0

In [ ]:
# Write the gains to the Scaler memory bank
pc.i.chan[0].SCALER.set_gain_table(test_gains_ramp[0], bank = 0, gain_timestamp = None, log_gain = None)

In [ ]:
# Read the gains from the Scaler memory bank

read_gains_ramp = np.array(pc.i.chan[0].SCALER.get_gain_table(bank = 0)) & 0xFFFF
read_gains_common = 2**pc.i.chan[0].SCALER.SHIFT_LEFT
read_gains_ramp_corrected = read_gains_common*((read_gains_ramp & 0x7FF) << (read_gains_ramp >> 11)).flatten()
read_gains_ramp_corrected

In [ ]:
plt.figure(figsize = (12, 8))

plt.subplot(411)
plt.title('verification of gain computation, gains read from memory')
plt.plot(pc.f, d_fft_ramp_mean_mag[0, bin_map])
plt.ylabel('|FFT|')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(412)
plt.semilogy(pc.f, test_gains_ramp[0, bin_map], label = 'gains (infinite resolution)')
plt.semilogy(pc.f, read_gains_ramp_corrected[bin_map], linestyle = '--', label = 'gains actual')
plt.ylabel('gains')
plt.xlim(0, 1600)
plt.grid()
plt.legend()

plt.subplot(413)
plt.semilogy(pc.f, np.abs(read_gains_ramp_corrected - test_gains_ramp[0])[bin_map])
plt.ylabel('$\Delta$gains')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(414)
plt.plot(pc.f[1:], (d_fft_ramp_mean_mag[0, bin_map][1:]*test_gains_ramp[0, bin_map][1:]/2**44) - test_target, label = 'gains (infinite resolution) $\\times$ |FFT|')
plt.plot(pc.f[1:], (d_fft_ramp_mean_mag[0, bin_map][1:]*read_gains_ramp_corrected[bin_map][1:]/2**44) - test_target, linestyle = 'dotted', label = 'gains actual $\\times$ |FFT|')
# plt.hlines(test_target, 0, 1600, color = 'red', linestyle = '--', label = 'target')
plt.ylabel('simulated \n|scaler| - target')
plt.xlabel('frequency [MHz]')
plt.xlim(0, 1600)
plt.grid()
plt.legend()

plt.tight_layout()

### Verify the Post-Gain Upper 16+16 Bits

In [ ]:
# # Check to make sure we're configured to receive channelizer data, not correlated data:
# if pc.ucap.OUTPUT_SOURCE_SEL != 0:
#     pc.ucap.OUTPUT_SOURCE_SEL = 0

# # Make sure we can get data from all channels:
# if pc.ucap.MODE != 0:
#     pc.ucap.MODE = 0 

# # Check that the FFT is not bypassed, the SCALER is bypassed, and set the data source as desired
# for n in range(pc.NPOLS):
#     pc.i.chan[n].FUNCGEN.set_data_source('buffer', ramp)
#     pc.i.chan[n].FFT.BYPASS = 1
#     pc.i.chan[n].SCALER.BYPASS = 1
#     pc.i.chan[n].SCALER.DATA_TYPE = 1 # Send the most significant 16+16 bits of the saturated post-gain FFT values
#     pc.i.chan[n].SCALER.CAP_DATA_TYPE = 2 # MSB of post-gain FFT value with saturation

# # Start data capture, grabbing data at the output of the
# # scaler, which has been bypassed.
# pc.i.start_data_capture(period = 0.02, 
#                          source = 'scaler'
#                          )

# # Capture data!
# t, d_post_gain_ramp, c = pc.u_receiver.read_raw_frames(format = '16+16', 
#                                                        ncap = 1, 
#                                                        split = True,
#                                                        verbose = 0
#                                                        )

In [ ]:
# Set the channelizer to capture post-gain data
pc.i.set_channelizer(adc_mode=None,
                     adc_sampling_mode=None,
                     adc_bandwidth=None,
                     adcdaq_mode=None,
                     data_source='buffer',
                     buffer = ramp,
                     function=None,
                     # funcgen_left_shift=0,
                     # funcgen_right_shift=0,
                     # freq_test_bins=None,  # will be passed into function_kwargs
                     fft_bypass=True,
                     fft_shift=0b00000000000000,
                     scaler_bypass=True,
                     scaler_cap_data_type=2,
                     scaler_bypass_data_type=None,
                     postscaler=None,
                     scaler_eight_bit=None,
                     scaler_rounding_mode=None,
                     symmetric_saturation=None,
                     zero_on_sat=None,
                     prober_user_flags=None,
                     offset_binary_encoding=None,
                     local_sync=True,
                     channels=list(np.arange(pc.NPOLS)),
                     gains = None # 1*2**32
                     )


# Start data capture, grabbing data at the output of the
# scaler, which has been bypassed.
pc.i.start_data_capture(period = 0.02, 
                        source = 'scaler',
                        mode = pc.ucap.MODE,
                        channels = list(np.arange(pc.NPOLS)),
                        data_type = 'fft_gain',
                        select = True
                        )

# Capture data!
t, d_post_gain_ramp, c = pc.u_receiver.read_raw_frames(format = '16+16', 
                                                       ncap = 1, 
                                                       split = True,
                                                       verbose = 0
                                                       )

In [ ]:
plt.figure(figsize = (12, 6))

plt.subplot(311)
plt.title('post-gain upper 16+16 bits, ramp')
plt.plot(pc.f, d_post_gain_ramp[0, 0, bin_map].real)
plt.ylabel('real')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(312)
plt.plot(pc.f, d_post_gain_ramp[0, 0, bin_map].imag)
plt.ylabel('imag')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(313)
plt.plot(pc.f, np.abs(d_post_gain_ramp[0, 0, bin_map]), label = 'measured')
plt.hlines(test_target*2**12, 0, 1600, color = 'C1', linestyle = '--', label = 'expected')
plt.ylabel('magnitude')
plt.xlabel('frequency [MHz]')
plt.xlim(0, 1600)
plt.grid()
plt.legend()

plt.tight_layout()

### Capture Scaler Data, Ramp

In [ ]:
# # Check to make sure we're configured to receive channelizer data, not correlated data:
# if pc.ucap.OUTPUT_SOURCE_SEL != 0:
#     pc.ucap.OUTPUT_SOURCE_SEL = 0

# # Make sure we can get data from all channels:
# if pc.ucap.MODE != 0:
#     pc.ucap.MODE = 0 

# # Check that the FFT is bypassed, the SCALER is not bypassed, and set the data source as desired
# for n in range(pc.NPOLS):
#     pc.i.chan[n].FUNCGEN.set_data_source('buffer', ramp)
#     pc.i.chan[n].FFT.BYPASS = 1
#     # if not pc.i.chan[n].SCALER.BYPASS:
#     pc.i.chan[n].SCALER.BYPASS = 0
#     # pc.i.chan[n].SCALER.DATA_TYPE = 0 # Send 4-bit scaler values
#     pc.i.chan[n].SCALER.CAP_DATA_TYPE = 2 # Main scaler output

# # Start data capture, grabbing data at the output of the
# # scaler, which has been bypassed.
# pc.i.start_data_capture(period = 0.02, 
#                          source = 'scaler'
#                          )

# t, d_sclr_ramp, c = pc.u_receiver.read_raw_frames(format = '16+16', 
#                                                    ncap = 1, 
#                                                    split = True,
#                                                    verbose = 0
#                                                    )

# d_sclr_ramp = d_sclr_ramp / 2**12

In [ ]:
# Set the channelizer to capture Scaler data
pc.i.set_channelizer(adc_mode=None,
                     adc_sampling_mode=None,
                     adc_bandwidth=None,
                     adcdaq_mode=None,
                     data_source='buffer',
                     buffer = ramp,
                     function=None,
                     # funcgen_left_shift=0,
                     # funcgen_right_shift=0,
                     # freq_test_bins=None,  # will be passed into function_kwargs
                     fft_bypass=True,
                     fft_shift=0b00000000000000,
                     scaler_bypass=False,
                     scaler_cap_data_type=0,
                     scaler_bypass_data_type=None,
                     postscaler=None,
                     scaler_eight_bit=None,
                     scaler_rounding_mode=None,
                     symmetric_saturation=None,
                     zero_on_sat=None,
                     prober_user_flags=None,
                     offset_binary_encoding=None,
                     local_sync=True,
                     channels=list(np.arange(pc.NPOLS)),
                     gains = None # 1*2**32
                     )


# Start data capture for 4+4-bit Scaler data
pc.i.start_data_capture(period = 0.02, 
                        source = 'scaler',
                        mode = pc.ucap.MODE,
                        channels = list(np.arange(pc.NPOLS)),
                        data_type = 0,
                        select = True
                        )

# Capture data!
t, d_sclr_ramp, c = pc.u_receiver.read_raw_frames(format = '16+16', 
                                                  ncap = 1, 
                                                  split = True,
                                                  verbose = 0
                                                  )

d_sclr_ramp = d_sclr_ramp / 2**12

In [ ]:
d_sclr_ramp_mean_mag = np.mean(np.abs(d_sclr_ramp[0]), axis = 0)
d_sclr_ramp_mean_mag

In [ ]:
plt.figure(figsize = (12, 4))
plt.plot(pc.f, d_sclr_ramp_mean_mag[bin_map], label = 'mean')
plt.hlines(test_target, 0, 1600, color = 'red', linestyle = '--', label = 'target')
plt.ylabel('scaler magnitude')
plt.xlabel('frequency [MHz]')
# plt.xlim(0, 1600)
plt.legend()
plt.grid()
plt.tight_layout()

In [ ]:
c_ramp_re, bin_edges = np.histogram(d_sclr_ramp[0].real.flatten(), bins = np.arange(-7.5, 7.5 + 1, 1))
c_ramp_im, bin_edges = np.histogram(d_sclr_ramp[0].imag.flatten(), bins = np.arange(-7.5, 7.5 + 1, 1))
bin_cents = np.arange(-7, 7 + 1)

plt.figure(figsize = (12, 4))

plt.subplot(121)
plt.title('real')
plt.bar(bin_cents, c_ramp_re/np.sum(c_ramp_re))
plt.vlines(-8, 0, np.max(c_ramp_re/np.sum(c_ramp_re)), color = 'black', linestyle = '--')
plt.vlines(8, 0, np.max(c_ramp_re/np.sum(c_ramp_re)), color = 'black', linestyle = '--')
plt.scatter(1, 1, color = 'red', label = 'expected')
plt.xticks(bin_cents)
plt.ylabel('normalized counts')
plt.xlabel('scaler LSBs')
plt.grid()
plt.legend()

plt.subplot(122)
plt.title('imag')
plt.bar(bin_cents, c_ramp_im/np.sum(c_ramp_im))
plt.vlines(-8, 0, np.max(c_ramp_im/np.sum(c_ramp_im)), color = 'black', linestyle = '--')
plt.vlines(8, 0, np.max(c_ramp_im/np.sum(c_ramp_im)), color = 'black', linestyle = '--')
plt.scatter(-1, 1, color = 'red', label = 'expected')
plt.xticks(bin_cents)
plt.xlabel('scaler LSBs')
plt.grid()

plt.tight_layout()

## Test with Real Signal

In [ ]:
target = 1.5*np.sqrt(2)
target_full = target*2**44

In [ ]:
d_fft_mag = np.mean(np.abs(d_fft), axis = 1)
gain = target_full/d_fft_mag

In [ ]:
plt.figure(figsize = (12, 6))

plt.subplot(311)
plt.semilogy(pc.f, np.sqrt(d_fft_mean_psd[0, bin_map]))
plt.ylabel('|FFT|')
plt.grid()

plt.subplot(312)
plt.semilogy(pc.f, gain[0, bin_map])
plt.ylabel('gains')
plt.grid()

plt.subplot(313)
plt.plot(pc.f, d_fft_mag[0, bin_map]*gain[0, bin_map]/2**44, label = 'gain $\\times$ |FFT|')
plt.hlines(target, 0, 1600, color = 'red', linestyle = '--', label = 'target')
plt.ylabel('|scaler| \n(simulated)')
plt.xlabel('frequency [MHz]')
plt.grid()
plt.legend()

plt.tight_layout()

In [ ]:
# ramp_gain = np.linspace(4e8, 6e8, 8192)[inv_bin_map]
# gain_set = 1511 << 10 # gain[0]/2**8
# gain_set = gain[0]/2**8|
# gain_set = 1e9*np.ones(pc.M//2)
# gain_set = 2**30*np.ones(pc.M//2)
# gain_set = 2**(np.floor(np.log2(6e8)))*np.ones(pc.M//2)
gain_set = gain[0]

# gain_set = glin[0]*2**glog[0]

In [ ]:
# # Gain breakdown for "old" gains
# trim_bits = 
# np.clip((np.ceil(np.log2(np.median(gain, axis = 1)))), 0, 31) - trim_bits

# # pc.i.chan[0].SCALER.set_gain_table(glin[0], bank=0, gain_timestamp=None, log_gain=int(glog[0]))
# # assert(pc.i.chan[0].SCALER.USE_FLOAT_GAINS == False)

# # Gain breakdown for floating point gains
# glin = gain[0] & 0x7FF
# glin

# glog = gain[0] >> 11

glog = (np.floor(np.log2(gain_set))-10).clip(0, 31)  #
glin = (gain_set / 2**glog).astype(np.uint16)
glog = glog.astype(np.uint16)
# print(f'{glog=}\n {gains=}')
# if any(gains < 0) or any(gains >= 2**11):
#     raise ValueError('Floating gain exceeds allowable range of 2^31*(2^11-1) = 4.3959E12')
# gains = (glog.astype(np.uint16) << 11) | gains.astype(np.uint16)
# print(f"Gains = {[f'{g:04x}' for g in gains[:10]]}")


# # ramp_gain = np.linspace(4e8, 6e8, 8192)[inv_bin_map]
# # gain_set = 1511 << 10 # gain[0]/2**8
# # gain_set = gain[0]/2**8|
# # gain_set = 1e9*np.ones(pc.M//2)
# # gain_set = gain[0]

In [ ]:
pc.i.chan[0].SCALER.set_gain_table(gain_set, bank = 0, gain_timestamp = None, log_gain = None)

In [ ]:
pc.i.chan[0].SCALER.USE_FLOAT_GAINS

In [ ]:
read_gains = np.array(pc.i.chan[0].SCALER.get_gain_table(bank = 0)) & 0xFFFF
read_gains_common = 2**pc.i.chan[0].SCALER.SHIFT_LEFT
read_gains_corrected = read_gains_common*((read_gains & 0x7FF) << (read_gains >> 11)).flatten()
read_gains_corrected

In [ ]:
read_gains & 0x7FF

In [ ]:
pc.i.chan[0].SCALER.SHIFT_LEFT

In [ ]:
(read_gains >> 11) + pc.i.chan[0].SCALER.SHIFT_LEFT

In [ ]:
read_gains_common

In [ ]:
plt.figure(figsize = (12, 6))

plt.subplot(211)
plt.title('computed gains compared with gains read from memory bank')
plt.semilogy(pc.f, gain_set[bin_map], label = 'gains, software')
# plt.semilogy(gain_set[bin_map])
plt.semilogy(pc.f, read_gains_corrected[bin_map], label = 'gains, actual')
# plt.plot(ramp_gain[bin_map])
plt.ylabel('gain')
plt.xlim(0, 1600)
plt.grid()
plt.legend()

plt.subplot(212)
plt.plot(pc.f, read_gains_corrected[bin_map] - gain_set[bin_map], color = 'black')
plt.ylabel('$\Delta$gain')
plt.xlabel('frequency [MHz]')
plt.xlim(0, 1600)
plt.grid()

plt.tight_layout()

In [ ]:
sim_sclr_mag = np.mean(np.abs(np.clip(np.round(d_fft[0]*read_gains_corrected/2**44), -7, 7)), axis = 0)

In [ ]:
plt.figure(figsize = (12, 8))

plt.subplot(311)
plt.semilogy(pc.f, np.sqrt(d_fft_mean_psd[0, bin_map]))
plt.ylabel('|FFT|')
plt.grid()

plt.subplot(312)
plt.semilogy(pc.f, read_gains_corrected[bin_map])
plt.ylabel('gains')
plt.grid()

plt.subplot(313)
plt.plot(pc.f, sim_sclr_mag[bin_map], label = 'gain, actual $\\times$ |FFT|')
plt.plot(pc.f, np.mean(np.abs(d_fft[0]*gain[0]/2**44), axis = 0)[bin_map], label = 'gain, software $\\times$ |FFT|')
plt.hlines(target, 0, 1600, color = 'red', linestyle = '--', label = 'target')
plt.ylabel('|scaler| \n(simulated)')
plt.xlabel('frequency [MHz]')
plt.grid()
plt.legend()

plt.tight_layout()

### Post-Gain 16+16 Real Data

In [ ]:
# # Check to make sure we're configured to receive channelizer data, not correlated data:
# if pc.ucap.OUTPUT_SOURCE_SEL != 0:
#     pc.ucap.OUTPUT_SOURCE_SEL = 0

# # Make sure we can get data from all channels:
# if pc.ucap.MODE != 0:
#     pc.ucap.MODE = 0 

# # Check that the FFT is not bypassed, the SCALER is bypassed, and set the data source as desired
# for n in range(pc.NPOLS):
#     pc.i.chan[n].FUNCGEN.set_data_source('adc')
#     pc.i.chan[n].FFT.BYPASS = 0
#     pc.i.chan[n].SCALER.BYPASS = 1
#     # pc.i.chan[n].SCALER.DATA_TYPE = 1 # Send the most significant 16+16 bits of the saturated post-gain FFT values
#     pc.i.chan[n].SCALER.CAP_DATA_TYPE = 2 # MSB of post-gain FFT value with saturation

# # Start data capture, grabbing data at the output of the
# # scaler, which has been bypassed.
# pc.i.start_data_capture(period = 0.02, 
#                          source = 'scaler'
#                          )

# # Capture data!
# t, d_post_gain, c = pc.u_receiver.read_raw_frames(format = '16+16', 
#                                                    ncap = 100, 
#                                                    split = True,
#                                                    verbose = 0
#                                                    )

In [ ]:
# Set the channelizer to capture post-gain data
pc.i.set_channelizer(adc_mode=None,
                     adc_sampling_mode=None,
                     adc_bandwidth=None,
                     adcdaq_mode=None,
                     data_source='adc',
                     function=None,
                     # funcgen_left_shift=0,
                     # funcgen_right_shift=0,
                     # freq_test_bins=None,  # will be passed into function_kwargs
                     fft_bypass=False,
                     fft_shift=0b00000000000000,
                     scaler_bypass=True,
                     scaler_cap_data_type=2,
                     scaler_bypass_data_type=None,
                     postscaler=None,
                     scaler_eight_bit=None,
                     scaler_rounding_mode=None,
                     symmetric_saturation=None,
                     zero_on_sat=None,
                     prober_user_flags=None,
                     offset_binary_encoding=None,
                     local_sync=True,
                     channels=list(np.arange(pc.NPOLS)),
                     gains = None # 1*2**32
                     )


# Start data capture, grabbing data at the output of the
# scaler, which has been bypassed.
pc.i.start_data_capture(period = 0.02, 
                        source = 'scaler',
                        mode = pc.ucap.MODE,
                        channels = list(np.arange(pc.NPOLS)),
                        data_type = 'fft_gain',
                        select = True
                        )

# Capture data!
t, d_post_gain, c = pc.u_receiver.read_raw_frames(format = '16+16', 
                                                  ncap = n_cap_fft//pc.n_frames_per_capture(), 
                                                  split = True,
                                                  verbose = 0
                                                  )

In [ ]:
plt.figure(figsize = (12, 6))

plt.subplot(311)
plt.plot(pc.f, d_post_gain[0, 0, bin_map].real)
plt.xlim(0, 1600)
plt.grid()

plt.subplot(312)
plt.plot(pc.f, d_post_gain[0, 0, bin_map].imag, color = 'C1')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(313)
plt.plot(pc.f, np.mean(np.abs(d_post_gain[0]), axis = 0)[bin_map], label = 'measured')
plt.plot(pc.f, d_fft_mag[0, bin_map]*read_gains_corrected[bin_map]/2**32, label = 'simulated, gains actual')
plt.xlim(0, 1600)
plt.legend()
plt.grid()

plt.tight_layout()

In [ ]:
d_post_gain_quant = np.clip(np.round(d_post_gain[0]/2**12), -7, 7)
d_post_gain_quant

In [ ]:
d_post_gain_quant_sim = np.clip(np.round(d_fft[0]*read_gains_corrected/2**44), -7, 7)
# d_post_gain_quant_sim = d_fft[0]*read_gains_corrected/2**44

In [ ]:
plt.figure(figsize = (12, 4))
plt.title('expected 4+4 bit output, using post-gain data')
plt.plot(pc.f, np.mean(np.abs(d_post_gain_quant), axis = 0)[bin_map], label = 'measured, post-gain')
plt.plot(pc.f, np.mean(np.abs(d_post_gain_quant_sim), axis = 0)[bin_map], label = 'simulated, gain actual')
plt.hlines(target, 0, 1600, color = 'red', linestyle = '--', label = 'target')
# plt.hlines(0, 0, 1600, color = 'black', linestyle = '--')
# plt.hlines(7*np.sqrt(2), 0, 1600, color = 'black', linestyle = '--')
plt.ylabel('|scaler|')
plt.xlabel('frequency [MHz]')
plt.xlim(0, 1600)
plt.grid()
plt.legend()
plt.tight_layout()

### Capture Scaler Data

In [ ]:
n_cap_scaler = n_cap_fft

In [ ]:
# # Check to make sure we're configured to receive channelizer data, not correlated data:
# if pc.ucap.OUTPUT_SOURCE_SEL != 0:
#     pc.ucap.OUTPUT_SOURCE_SEL = 0

# # Make sure we can get data from all channels:
# if pc.ucap.MODE != 0:
#     pc.ucap.MODE = 0 

# # Check that the FFT and Scaler are not bypassed, and set the data source as desired
# for n in range(pc.NPOLS):
#     pc.i.chan[n].FUNCGEN.set_data_source('adc')
#     pc.i.chan[n].FFT.BYPASS = 0
#     pc.i.chan[n].SCALER.BYPASS = 0
#     # pc.i.chan[n].SCALER.DATA_TYPE = 0 # Send 4-bit scaler values
#     pc.i.chan[n].SCALER.CAP_DATA_TYPE = 2 # Main scaler output

# # Start data capture, grabbing data at the output of the
# # scaler, which has been bypassed.
# pc.i.start_data_capture(period = 0.02, 
#                          source = 'scaler'
#                          )

# # Capture data!
# t, d_sclr, c = pc.u_receiver.read_raw_frames(format = '16+16', 
#                                              ncap = n_cap_scaler//pc.n_frames_per_capture(), 
#                                              split = True,
#                                              verbose = 0
#                                              )

# d_sclr = d_sclr / 2**12

In [ ]:
# Set the channelizer to capture Scaler data
pc.i.set_channelizer(adc_mode=None,
                     adc_sampling_mode=None,
                     adc_bandwidth=None,
                     adcdaq_mode=None,
                     data_source='adc',
                     function=None,
                     # funcgen_left_shift=0,
                     # funcgen_right_shift=0,
                     # freq_test_bins=None,  # will be passed into function_kwargs
                     fft_bypass=False,
                     fft_shift=0b00000000000000,
                     scaler_bypass=False,
                     scaler_cap_data_type=0,
                     scaler_bypass_data_type=None,
                     postscaler=None,
                     scaler_eight_bit=None,
                     scaler_rounding_mode=None,
                     symmetric_saturation=None,
                     zero_on_sat=None,
                     prober_user_flags=None,
                     offset_binary_encoding=None,
                     local_sync=True,
                     channels=list(np.arange(pc.NPOLS)),
                     gains = None # 1*2**32
                     )


# Start data capture for 4+4-bit Scaler data
pc.i.start_data_capture(period = 0.02, 
                        source = 'scaler',
                        mode = pc.ucap.MODE,
                        channels = list(np.arange(pc.NPOLS)),
                        data_type = 0,
                        select = True
                        )

# Capture data!
t, d_sclr, c = pc.u_receiver.read_raw_frames(format = '16+16', 
                                             ncap = n_cap_scaler//pc.n_frames_per_capture(), 
                                             split = True,
                                             verbose = 0
                                             )

d_sclr = d_sclr / 2**12

In [ ]:
d_sclr_mean_mag = np.mean(np.abs(d_sclr[0]), axis = 0)
d_sclr_mean_mag

In [ ]:
plt.figure(figsize = (12, 6))
plt.title(f'scaler output to target ratio: {np.round(np.mean(d_sclr_mean_mag)/target, 2)}')
plt.plot(pc.f, d_sclr_mean_mag[bin_map], label = 'mean magnitude, scaler')
plt.plot(pc.f, sim_sclr_mag, label = 'simulated magnitude, gains actual $\\times$ |FFT|')
# plt.plot(pc.f, np.sqrt(np.mean(np.abs(d_post_gain[0]), axis = 0)[bin_map])/2**12), label = 'average magnitude, post-gain')
# plt.plot(pc.f, d_sclr_mean_mag[bin_map]/2, label = 'average magnitude / 2')
plt.hlines(target, 0, 1600, color = 'red', linestyle = '--', label = 'target')
# plt.hlines(0, 0, 1600, color = 'black', linestyle = '--')
# plt.hlines(7*np.sqrt(2), 0, 1600, color = 'black', linestyle = '--')
plt.ylabel('|scaler|')
plt.xlabel('frequency [MHz]')
plt.xticks(np.arange(0, 1600 + 100, 100))
plt.xlim(0, 1600)
plt.grid()
plt.legend()
plt.tight_layout()

### Generate Scaler Histograms

In [ ]:
d_sclr[0].shape

In [ ]:
c_re, bin_edges = np.histogram(d_sclr[0].real.flatten(), bins = np.arange(-7.5, 7.5 + 1, 1))
c_im, bin_edges = np.histogram(d_sclr[0].imag.flatten(), bins = np.arange(-7.5, 7.5 + 1, 1))
bin_cents = np.arange(-7, 7 + 1)

In [ ]:
plt.figure(figsize = (12, 4))

plt.subplot(121)
plt.title('real')
plt.bar(bin_cents, c_re/np.sum(c_re))
plt.vlines(-8, 0, np.max(c_re/np.sum(c_re)), color = 'black', linestyle = '--')
plt.vlines(8, 0, np.max(c_re/np.sum(c_re)), color = 'black', linestyle = '--')
plt.xticks(bin_cents)
plt.ylabel('normalized counts')
plt.xlabel('scaler LSBs')
plt.grid()

plt.subplot(122)
plt.title('imag')
plt.bar(bin_cents, c_im/np.sum(c_im))
plt.vlines(-8, 0, np.max(c_im/np.sum(c_im)), color = 'black', linestyle = '--')
plt.vlines(8, 0, np.max(c_im/np.sum(c_im)), color = 'black', linestyle = '--')
plt.xticks(bin_cents)
plt.xlabel('scaler LSBs')
plt.grid()

plt.tight_layout()

### Convert Scaler Units to FFT Units

In [ ]:
scaler_to_fft = 2**44/read_gains_corrected # gain_set # gain[0]
d_sclr_fft_units = scaler_to_fft*d_sclr_mean_mag

In [ ]:
plt.figure(figsize = (12, 6))

# plt.subplot(211)
plt.title('comparison of FFT magnitude and Scaler magnitude')
plt.semilogy(pc.f, d_fft_mag[0, bin_map], label = 'FFT')
plt.semilogy(pc.f, d_sclr_fft_units[bin_map], label = 'scaler')
plt.ylabel('magnitude [FFT units]')
plt.xlabel('frequency [MHz]')
plt.xlim(0, 1600)
plt.xticks(np.arange(0, 1600 + 100, 100))
plt.grid()
plt.legend()

plt.tight_layout()

# Use Digital Gains to Captured Correlated Data

Recall that the integrated autocorrelation of a frequency-domain signals $X(f)$ is

\begin{equation}
    A = \Sigma_{i = 0}^{N - 1}(X_i X_i^*),
\end{equation}

and the cross-correlation of two frequency-domain signals $X(f)$ and $Y(f)$ is 

\begin{equation}
    C = \Sigma_{i = 0}^{N - 1}(X_i Y_i^*).
\end{equation}

We can make direct comparisons of the raw FFT and Scaler data with the correlated data, so long as we remember that we need to make sure that we integrate the *power* of the raw FFT and Scaler data! The correlator, in its simplest form, is multiplying two frequency bins together for $N$ frames, and adding those values up.  

(I make this note since I once caused myself an inordinate amount of confusion by simply comparing to the square of the integrated magnitude of the raw FFT and Scaler data, which is not the same mathematical operation!) 

## Start with FUNCGEN Ramp Correlations 

In [ ]:
# pc.i.chan[1].SCALER.set_gain_table(gain[1], bank = 0, gain_timestamp = None, log_gain = None)

In [ ]:
# Reset the gains for real data
test_gains_ramp[np.where(gain == np.inf)] = 1e-10
for n in range(pc.NPOLS):
    pc.i.chan[n].SCALER.set_gain_table(test_gains_ramp[0], bank = 0, gain_timestamp = None, log_gain = None)

In [ ]:
# Write the gains to the Scaler memory bank
# pc.i.chan[0].SCALER.set_gain_table(test_gains_ramp[0], bank = 0, gain_timestamp = None, log_gain = None)
# pc.i.chan[1].SCALER.set_gain_table(test_gains_ramp[0], bank = 0, gain_timestamp = None, log_gain = None)

# Set the channelizer to capture Scaler data
pc.i.set_channelizer(adc_mode=None,
                     adc_sampling_mode=None,
                     adc_bandwidth=None,
                     adcdaq_mode=None,
                     data_source='buffer',
                     buffer = ramp,
                     function=None,
                     # funcgen_left_shift=0,
                     # funcgen_right_shift=0,
                     # freq_test_bins=None,  # will be passed into function_kwargs
                     fft_bypass=True,
                     fft_shift=0b00000000000000,
                     scaler_bypass=False,
                     scaler_cap_data_type=0,
                     scaler_bypass_data_type=None,
                     postscaler=None,
                     scaler_eight_bit=None,
                     scaler_rounding_mode=None,
                     symmetric_saturation=None,
                     zero_on_sat=None,
                     prober_user_flags=None,
                     offset_binary_encoding=None,
                     local_sync=True,
                     channels=list(np.arange(pc.NPOLS)),
                     gains = None # 1*2**32
                     )

In [ ]:
# # Check to make sure we're configured to receive correlator data, not channelizer data:
# if pc.ucap.OUTPUT_SOURCE_SEL != 1:
#     pc.ucap.OUTPUT_SOURCE_SEL = 1

# # Make sure we can get data from all channels:
# if pc.ucap.MODE != 0:
#     pc.ucap.MODE = 0 

# # Check that the FFT and Scaler are not bypassed, and set the data source as desired
# for n in range(pc.NPOLS):
#     pc.i.chan[n].FUNCGEN.set_data_source('adc')
#     pc.i.chan[n].FFT.BYPASS = 0
#     pc.i.chan[n].SCALER.BYPASS = 0
#     pc.i.chan[n].SCALER.DATA_TYPE = 0 # Send 4-bit scaler values

# if not pc.corr_is_configured:
#     pc.configure_corr()

# Configure the parameters of the correlator
pc.corr = pc.i.CORR # UCORR object
pc.corr_receiver = pc.i.get_corr_receiver() # instantiate a correlator receiver object
# pc.corr.AUTOCORR_ONLY = 0 
# pc.corr.NO_ACCUM = 0 
# pc.corr.INTEGRATION_PERIOD = 16384 - 1

n_firmware_frames = 16384
pc.NCORR_PROD = int(pc.NPOLS*(pc.NPOLS + 1) // 2)

pc.i.start_correlator(integration_period = n_firmware_frames)

In [ ]:
s_per_frame = 2**14*(1/(pc.fs*1e6)) # samples/frame * seconds/sample
n_software_frames = 119
number_of_results = 1

In [ ]:
pc.generate_prod_map()

In [ ]:
t1 = time.time()

_, d_corr_ramp, counts, sat = pc.corr_receiver.read_corr_frames(soft_integ_period = n_software_frames,
                                                              number_of_results = number_of_results,
                                                              flush = True,
                                                              data_timeout = 0.1,
                                                              flush_timeout = 0.01,
                                                              return_format = 'raw',
                                                              verbose = 1
                                                              )

t2 = time.time()

print(f'Took {t2 - t1} s to recover data')

In [ ]:
print(pc.corr_receiver.NCMAC, pc.corr_receiver.NPROD, pc.corr_receiver.NCLK, pc.corr_receiver.NCHAN)

In [ ]:
pc.corr.OVERRUN

In [ ]:
d_corr_ramp.shape

In [ ]:
np.abs(d_corr_ramp[0, :, 0])

In [ ]:
counts.shape

In [ ]:
counts

### To-Debug 

I don't think the sat values are actually reflective of any saturations in the firmware accumulator. If the Scaler magnitude is high (for example, with a mean magnitude of 3.5 or greater), the firmware accumulator will saturate, and the output will be a flat line, but the sat array will return all 0+0j.

In [ ]:
sat

In [ ]:
d_corr_ramp.shape

In [ ]:
plt.figure(figsize = (12, 4))
plt.plot(pc.f, np.abs(d_corr_ramp[0, :, 0]), label = 'measured')
plt.hlines(test_target**2*n_firmware_frames*n_software_frames, pc.f[0], pc.f[-1], color = 'red', linestyle = '--', label = 'target')
plt.ylabel('raw correlator output')
plt.xlabel('frequency [MHz]')
plt.grid()
plt.legend()
plt.tight_layout()

In [ ]:
# OLD

# corr_to_sclr_units = 1/(np.unique(counts[0, 0])[0]*n_firmware_frames)
# d_corr_sclr_units = np.abs(d_corr[0, 0, :, 0]*corr_to_sclr_units) # take abs just to recast to float

In [ ]:
counts[0]

In [ ]:
counts_good = counts[0] == n_software_frames*len(pc.f)

if counts_good:
    corr_to_sclr_units = 1/(n_software_frames*n_firmware_frames)
    d_corr_ramp_sclr_units = d_corr_ramp*corr_to_sclr_units # take abs just to recast to float
else:
    print(counts)
    raise ValueError('Check your counts array. You may have dropped packets.')

In [ ]:
d_sclr_ramp_mean_psd = np.mean(np.abs(d_sclr_ramp[0])**2, axis = 0)

In [ ]:
plt.figure(figsize = (12, 6))
# plt.plot(pc.f, d_sclr_mean_psd[bin_map], label = 'scaler magnitude')
plt.plot(pc.f, d_sclr_ramp_mean_psd[bin_map], label = 'scaler mean power')
plt.plot(pc.f, d_corr_ramp_sclr_units[0, :, 0], label = 'correlated data, scaler units')
# plt.hlines(target**2, 0, 1600, color = 'red', linestyle = '--', label = 'target')
plt.ylabel('scaler autocorrelation')
plt.xlabel('frequency [MHz]')
plt.grid()
plt.legend()
plt.tight_layout()

In [ ]:
d_corr_ramp_fft_units = np.zeros(d_corr_ramp.shape, dtype = np.complex128)

# correct gains for zeros
read_gains_ramp_corrected[np.where(read_gains_ramp_corrected == 0)[0]] = 1

for n in range(pc.NCORR_PROD):
    p0 = pc.prod[n][0]
    p1 = pc.prod[n][1]

    corr_ramp_sclr_units_to_corr_fft_units = (2**44)**2/(read_gains_ramp_corrected[bin_map]**2)
    d_corr_ramp_fft_units[..., n] = d_corr_ramp_sclr_units[..., n]*corr_ramp_sclr_units_to_corr_fft_units[None, :]

In [ ]:
d_fft_ramp_mean_psd = np.mean(np.abs(d_fft_ramp)**2, axis = 1)
d_sclr_ramp_mean_psd_fft_units = np.mean(np.abs(2**44*d_sclr_ramp[0]/read_gains_ramp_corrected)**2, axis = 0)

In [ ]:
d_corr_ramp_fft_units

In [ ]:
plt.figure(figsize = (12, 8))

plt.subplot(211)
plt.title('comparison of FFT, scaler, and autocorrelation spectra')
plt.semilogy(pc.f, d_fft_ramp_mean_psd[0, bin_map], label = 'FFT')
plt.semilogy(pc.f, d_sclr_ramp_mean_psd_fft_units[bin_map], label = 'scaler')
plt.semilogy(pc.f, d_corr_ramp_fft_units[0, :, 0], label = 'auto')
plt.ylabel('power [arb.]')
plt.xticks(np.arange(0, 1600 + 100, 100))
plt.grid()
plt.legend(loc = 'lower right')

plt.subplot(212)
plt.semilogy(pc.f[1:], (d_sclr_ramp_mean_psd_fft_units[bin_map] - d_fft_ramp_mean_psd[0, bin_map])[1:], label = 'scaler', color = 'C1')
plt.semilogy(pc.f[1:], (d_corr_ramp_fft_units[0, :, 0] - d_fft_ramp_mean_psd[0, bin_map])[1:], label = 'corr', color = 'C2')
plt.ylabel('residuals [arb.]')
plt.xlabel('frequency [MHz]')
plt.xticks(np.arange(0, 1600 + 100, 100))
plt.grid()

plt.tight_layout()

## Validate Correlations of Real Input Data

In [ ]:
# Reset the gains for real data
gain[np.where(gain == np.inf)] = 0
for n in range(pc.NPOLS):
    pc.i.chan[n].SCALER.set_gain_table(gain[n], bank = 0, gain_timestamp = None, log_gain = None)

In [ ]:
pc.i.set_channelizer(adc_mode=None,
                     adc_sampling_mode=None,
                     adc_bandwidth=None,
                     adcdaq_mode=None,
                     data_source='adc',
                     function=None,
                     funcgen_left_shift=0,
                     funcgen_right_shift=0,
                     # freq_test_bins=None,  # will be passed into function_kwargs
                     fft_bypass=False,
                     fft_shift=0b00000000000000,
                     scaler_bypass=False,
                     scaler_cap_data_type=None,
                     scaler_bypass_data_type=None,
                     postscaler=None,
                     scaler_eight_bit=False,
                     scaler_rounding_mode=2,
                     symmetric_saturation=True,
                     zero_on_sat=False,
                     prober_user_flags=None,
                     offset_binary_encoding=False,
                     local_sync=True,
                     channels=list(np.arange(pc.NPOLS)),
                     gains = None # 1*2**32
                     )

In [ ]:
pc.i.start_correlator(integration_period = n_firmware_frames)

In [ ]:
n_software_frames = 1

t1 = time.time()

_, d_corr, counts, sat = pc.corr_receiver.read_corr_frames(soft_integ_period = n_software_frames,
                                                         number_of_results = number_of_results,
                                                         flush = True,
                                                         data_timeout = 0.1,
                                                         flush_timeout = 0.01,
                                                         return_format = 'raw',
                                                         verbose = 1
                                                         )

t2 = time.time()

print(f'Took about {t2 - t1} s to recover data')

In [ ]:
plt.figure(figsize = (12, 4))
plt.plot(pc.f, np.abs(d_corr[0, :, 0]), label = 'measured')
# plt.hlines(target**2*n_firmware_frames*n_software_frames, pc.f[0], pc.f[-1], color = 'red', linestyle = '--', label = 'target')
plt.hlines(n_software_frames*2**17, pc.f[0], pc.f[-1], color = 'black', ls = '--', label = 'limit')
plt.ylabel('raw correlator output')
plt.xlabel('frequency [MHz]')
plt.grid()
plt.legend()
plt.tight_layout()

In [ ]:
counts_good = counts[0] == n_software_frames*len(pc.f)

if counts_good:
    corr_to_sclr_units = 1/(n_software_frames*n_firmware_frames)
    d_corr_sclr_units = d_corr*corr_to_sclr_units
else:
    print(counts)
    raise ValueError('Check your counts array. You may have dropped packets.')

In [ ]:
d_sclr_mean_psd = np.mean(np.abs(d_sclr[0])**2, axis = 0)

In [ ]:
plt.figure(figsize = (12, 6))
plt.plot(pc.f, d_sclr_mean_psd[bin_map], label = 'scaler mean power')
plt.plot(pc.f, np.abs(d_corr_sclr_units[0, :, 0]), label = 'correlated data, scaler units')
# plt.hlines(target**2, 0, 1600, color = 'red', linestyle = '--', label = 'target')
plt.ylabel('scaler autocorrelation')
plt.xlabel('frequency [MHz]')
plt.grid()
plt.legend()
plt.tight_layout()

In [ ]:
read_gains_corrected = np.zeros((pc.NPOLS, pc.NBINS))

for n in range(pc.NPOLS):
    read_gains = np.array(pc.i.chan[n].SCALER.get_gain_table(bank = 0)) & 0xFFFF
    read_gains_common = 2**pc.i.chan[n].SCALER.SHIFT_LEFT
    read_gains_corrected[n] = read_gains_common*((read_gains & 0x7FF) << (read_gains >> 11)).flatten()

read_gains_corrected[np.where(read_gains_corrected) == 0.] = np.nan

In [ ]:
pc.generate_prod_map()

In [ ]:
d_corr_fft_units = np.zeros(d_corr.shape, dtype = np.complex128)

for n in range(pc.NCORR_PROD):
    p0 = pc.prod[n][0]
    p1 = pc.prod[n][1]

    try:
        corr_sclr_units_to_corr_fft_units = (2**44)**2/(read_gains_corrected[p0, bin_map]*read_gains_corrected[p1, bin_map])
        d_corr_fft_units[..., n] = d_corr_sclr_units[..., n]*corr_sclr_units_to_corr_fft_units[None, :]
    except:
        continue

In [ ]:
d_fft_mean_psd = np.mean(np.abs(d_fft)**2, axis = 1)
d_sclr_mean_psd_fft_units = np.mean(np.abs(2**44*d_sclr[0]/read_gains_corrected[:, None, :])**2, axis = 1)

In [ ]:
d_corr_fft_units.shape

In [ ]:
d_sclr_mean_psd_fft_units.shape

In [ ]:
plt.figure(figsize = (12, 8))

plt.subplot(211)
plt.title('comparison of FFT, scaler, and autocorrelation spectra')
plt.semilogy(pc.f, d_fft_mean_psd[0, bin_map], label = 'FFT')
plt.semilogy(pc.f, d_sclr_mean_psd_fft_units[0, bin_map], label = 'scaler')
plt.semilogy(pc.f, d_corr_fft_units[0, :, 0], label = 'auto')
plt.ylabel('power [arb.]')
plt.xticks(np.arange(0, 1600 + 100, 100))
plt.grid()
plt.legend(loc = 'lower right')

plt.subplot(212)
plt.plot(pc.f[1:], (d_sclr_mean_psd_fft_units[0, bin_map] - d_fft_mean_psd[0, bin_map])[1:], label = 'scaler', color = 'C1')
plt.plot(pc.f[1:], (d_corr_fft_units[0, :, 0] - d_fft_mean_psd[0, bin_map])[1:], label = 'corr', color = 'C2')
plt.ylabel('residuals [arb.]')
plt.xlabel('frequency [MHz]')
plt.xticks(np.arange(0, 1600 + 100, 100))
plt.grid()

plt.tight_layout()

In [ ]:
plt.figure(figsize = (12, 8))

plt.subplot(211)
plt.title('comparison of FFT, scaler, and autocorrelation spectra')
plt.semilogy(pc.f, d_fft_mean_psd[0, bin_map], label = 'FFT')
plt.semilogy(pc.f, d_sclr_mean_psd_fft_units[0, bin_map], label = 'scaler')
plt.semilogy(pc.f, d_corr_fft_units[0, :, 0], label = 'auto')
plt.ylabel('power [arb.]')
plt.xticks(np.arange(0, 1600 + 100, 100))
plt.grid()
plt.legend(loc = 'lower right')

plt.subplot(212)
plt.plot(pc.f[1:], 100*((d_sclr_mean_psd_fft_units[0, bin_map] - d_fft_mean_psd[0, bin_map])/d_fft_mean_psd[0, bin_map])[1:], label = 'scaler', color = 'C1')
plt.plot(pc.f[1:], 100*((d_corr_fft_units[0, :, 0] - d_fft_mean_psd[0, bin_map])/d_fft_mean_psd[0, bin_map])[1:], label = 'corr', color = 'C2')
plt.ylabel('fractional deviation \nfrom raw FFT [%]')
plt.xlabel('frequency [MHz]')
plt.xticks(np.arange(0, 1600 + 100, 100))
plt.grid()

plt.tight_layout()

In [ ]:
plt.figure(figsize = (12, 6))

plt.title('comparison of FFT, scaler, and autocorrelation spectra')
plt.semilogy(pc.f, d_fft_mean_psd[0, bin_map], label = 'FFT')
plt.semilogy(pc.f, d_corr_fft_units[0, :, 0], color = 'C2', label = 'auto')
plt.ylim(3e7, 6e7)
plt.xlim(100, 150)
# plt.xticks(np.arange(0, 1600 + 100, 100))
plt.ylabel('power [arb.]')
plt.xlabel('frequency [MHz]')
plt.grid()
plt.legend(loc = 'lower right')
plt.tight_layout()